# Amazon ML Challenge 2026 — Business Entity Resolution, submission 3a3 (Kaggle, 2× T4)

**Submission 3a3 = submission 2 (`erk_sub2_1`) + one change:** `train_s1_frac` 0.5 instead of 0.3 when the extra memory it needs fits (checked just before the feature matrix is built; otherwise 0.3, the setting proven on Kaggle). Look for the `train_s1_frac` lines in the training log. Benchmark (KY+KL, 3 runs): +0.0020 macro F0.5 over submission 2.

**Settings:** Accelerator **GPU T4 ×2**, Internet **On** (for `pip`), dataset attached
(the folder containing `dataset/train`, `dataset/test`, `utils/`). Use **Save Version → Save & Run All**.
When it finishes, `/kaggle/working` contains only **`<team>_submission.zip`**:
`output/matching_results.tsv` inside it is the leaderboard file; the zip itself is the final package.

| stage | what | where it runs |
| --- | --- | --- |
| 1 normalise | Indic romanisation (9 scripts, one table) + phonetic skeleton; legal forms, DBA, domains, leet typos, PMB, US/IN/FR states | CPU |
| 2 regions | missing S2/S3 states filled from a token→state table learned from S1 (99.998% accurate on France) | CPU |
| 3 block | S2/S3 → S1 top-k IDF-cosine per (country, state); wide name-only channel for address-less records | CPU |
| 4 features | name/address/number similarities, name rarity + orphan pressure, competitor context | CPU |
| 5 score | XGBoost stage 1 (3-fold OOF, folds in parallel on both GPUs) → stage 2 (2 boosters averaged) with sibling features | **GPU** |
| 6 decide | one S1 per S2/S3 record; per-S1 set maximising expected F0.5 | CPU |

**Changes since submission 1 (LB 0.9687), all measured on train before adopting:**
test-like training/validation (15% of train S1 orphaned → ~38% distractors, test ~34–39%);
region inference; blocking parameters from a recall sweep; orphan-pressure name features;
S2↔S3 sibling features; 2-booster stage 2; per-segment reporting; French ordinal fix.

## 0. Environment

In [ ]:
!pip install -q "polars==1.44.2" "rapidfuzz==3.14.6" "sparse_dot_topn==1.2.0" "xgboost==3.4.1" psutil
!nvidia-smi -L
import importlib, platform
for m in ["polars", "rapidfuzz", "sparse_dot_topn", "xgboost", "numpy", "scipy", "psutil"]:
    print(f"{m:16s}", importlib.import_module(m).__version__)
import xgboost, os
print("python", platform.python_version(), "| xgboost CUDA build:", xgboost.build_info().get("USE_CUDA"),
      "| CPUs:", os.cpu_count())

In [ ]:
import os, sys, json, shutil, subprocess, zipfile, datetime
from pathlib import Path

# ============================================================================
# DEV = False -> the real, full-dataset run. This IS the submission run.
# DEV = True  -> smoke test on a slice of states (~15-25 min). Output is NOT
#                a valid submission (most test S1 rows are excluded on purpose).
# ============================================================================
DEV = False
TEAM = "team"          # <- your team name (used for the zip and the write-up)
MEMBERS = "Pranav Kishan T Y, Narain B K, Anurup R Krishnan, Sanjeev Srinivas"

IS_KAGGLE = Path("/kaggle/input").exists()
BASE = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()

def find_dataset():
    """Search for a dataset/ folder with both train/ and test/ source1 files
    (not just train/) so a partial or mismatched dataset is never silently accepted."""
    roots = [Path("/kaggle/input")] if IS_KAGGLE else [Path.cwd(), Path.cwd().parent]
    roots = [r for r in roots if r.exists()]
    hits = []
    for r in roots:
        for p in r.rglob("train_source1.tsv"):
            data_dir = p.parent.parent
            if (data_dir / "test" / "test_source1.tsv").exists():
                hits.append(data_dir)
    if not hits:
        seen = [str(x) for r in roots for x in list(r.rglob("*.tsv"))[:20]]
        raise FileNotFoundError(
            f"No dataset/ folder with BOTH train/train_source1.tsv and test/test_source1.tsv found "
            f"under {[str(r) for r in roots]}.\n"
            f".tsv files actually visible there: {seen or 'NONE - is the dataset attached? (Add Input, top right)'}"
        )
    hits = sorted(set(hits), key=lambda p: len(p.parts))   # prefer the shallowest match
    if len(hits) > 1:
        print(f"WARNING: {len(hits)} candidate dataset folders found, using the shallowest:")
        for h in hits:
            print("  ", h)
    return hits[0]

DATA_DIR = find_dataset()
RESOURCE_DIR = DATA_DIR.parent                # student_resource/ (utils/validate_submission.py)

# Fail loudly and immediately, not 10 minutes into normalisation, if anything is missing.
required = [DATA_DIR / "train" / f"train_{f}.tsv" for f in ("source1", "source2", "source3", "ground_truth")]
required += [DATA_DIR / "test" / f"test_{f}.tsv" for f in ("source1", "source2", "source3")]
required += [RESOURCE_DIR / "utils" / "validate_submission.py"]
missing = [str(p) for p in required if not p.exists()]
assert not missing, f"Missing required files, fix the dataset attachment before continuing:\n" + "\n".join(missing)
print(f"OK: found dataset at {DATA_DIR}")
print(f"OK: found validator at {RESOURCE_DIR / 'utils' / 'validate_submission.py'}")

CODE_DIR = BASE / "code" / "business_entity_resolution"
SRC_DIR = CODE_DIR / "src"
(SRC_DIR / "ber").mkdir(parents=True, exist_ok=True)
CFG = {
    "data_dir": str(DATA_DIR),
    "work_dir": "/tmp/ber_work" if IS_KAGGLE else str(BASE / "work"),   # caches, not packaged
    "output_dir": str(BASE / "output"),
    "workers": None,                   # normalisation processes: all cores
    "dev_states": ["us_tx", "in_ka", "fr_naq"] if DEV else None,
    # share of non-holdout train S1 entities used to fit. 0.5 OOM'd on the full run (30M-row
    # feature matrix, concurrent GPU fold/variant training doubling peak RAM via boolean-index
    # copies). model.py now trains folds/variants sequentially (that alone should fix it), and
    # 0.3 is a large extra safety margin on top; raise it once a run succeeds with real headroom
    # left in the RAM lines the log prints at each step.
    "train_s1_frac": 0.3,
    # [sub3a3] 0.5 is used when its extra memory over 0.3 fits (see choose_train_frac in pipeline.py)
    "train_s1_frac_options": [0.5, 0.3],
    # share of train S1 entities removed so their S2/S3 records act as distractors (test-like)
    "orphan_frac": 0.15,
    "stage1_rounds": 3000,
    "stage2_rounds": 3000,
    "xgb_params": {"learning_rate": 0.05},
}
print()
print("*" * 70)
print("*** DEV SMOKE TEST — output is NOT a valid submission ***" if DEV else
      "*** FULL RUN — this IS the real submission run ***")
print("*" * 70)
print(json.dumps(CFG, indent=2))

## 1. Pipeline source → `code/business_entity_resolution/src/`

In [ ]:
%%writefile "{SRC_DIR}/ber/__init__.py"
"""Business entity resolution pipeline (Amazon ML Challenge 2026)."""

In [ ]:
%%writefile "{SRC_DIR}/ber/normalize.py"
"""Text normalisation for business names and addresses.

Everything here is pure Python and deterministic, so it can run in a
multiprocessing pool over ~22M records. Nothing is looked up externally: the
only domain knowledge is abbreviation/state tables written by hand.

Key ideas
- Indic scripts (Devanagari, Bengali, Gurmukhi, Gujarati, Oriya, Tamil, Telugu,
  Kannada, Malayalam) share one Unicode layout, so every script is mapped onto
  the Devanagari block by code-point offset and romanised with a single table.
- Hindi business names are phonetic spellings of English words
  (e.g. "कंसल्टेंट्स" -> "kansaltents" for "consultants"), so exact
  transliteration never lines up. `phonetic_key` reduces both sides to a
  consonant skeleton ("knsltnts") that does.
"""
import re
import unicodedata

# ---------------------------------------------------------------------------
# Indic -> Latin
# ---------------------------------------------------------------------------
_INDIC_BLOCKS = [0x0900, 0x0980, 0x0A00, 0x0A80, 0x0B00, 0x0B80, 0x0C00, 0x0C80, 0x0D00]

# Offsets inside a 128-codepoint Indic block (Devanagari layout).
_CONSONANTS = {
    0x15: "k", 0x16: "kh", 0x17: "g", 0x18: "gh", 0x19: "n",
    0x1A: "ch", 0x1B: "chh", 0x1C: "j", 0x1D: "jh", 0x1E: "n",
    0x1F: "t", 0x20: "th", 0x21: "d", 0x22: "dh", 0x23: "n",
    0x24: "t", 0x25: "th", 0x26: "d", 0x27: "dh", 0x28: "n", 0x29: "n",
    0x2A: "p", 0x2B: "ph", 0x2C: "b", 0x2D: "bh", 0x2E: "m",
    0x2F: "y", 0x30: "r", 0x31: "r", 0x32: "l", 0x33: "l", 0x34: "l",
    0x35: "v", 0x36: "sh", 0x37: "sh", 0x38: "s", 0x39: "h",
    0x58: "k", 0x59: "kh", 0x5A: "g", 0x5B: "z", 0x5C: "d", 0x5D: "dh", 0x5E: "f", 0x5F: "y",
}
_VOWELS = {
    0x04: "a", 0x05: "a", 0x06: "a", 0x07: "i", 0x08: "i", 0x09: "u", 0x0A: "u", 0x0B: "ri",
    0x0C: "li", 0x0D: "e", 0x0E: "e", 0x0F: "e", 0x10: "ai", 0x11: "o", 0x12: "o",
    0x13: "o", 0x14: "au", 0x60: "ri", 0x72: "e",
}
_SIGNS = {
    0x3E: "a", 0x3F: "i", 0x40: "i", 0x41: "u", 0x42: "u", 0x43: "ri", 0x44: "ri",
    0x45: "e", 0x46: "e", 0x47: "e", 0x48: "ai", 0x49: "o", 0x4A: "o", 0x4B: "o",
    0x4C: "au", 0x62: "li", 0x63: "li",
}
_MODIFIERS = {0x01: "n", 0x02: "\x02", 0x03: "h"}
_VIRAMA, _NUKTA = 0x4D, 0x3C
_NUKTA_MAP = {"j": "z", "ph": "f", "k": "k", "kh": "kh", "g": "g", "d": "d", "dh": "dh"}


def _indic_offset(ch):
    cp = ord(ch)
    if 0x0900 <= cp <= 0x0D7F:
        return cp - (cp & ~0x7F)
    return None


def has_indic(text):
    return any(0x0900 <= ord(c) <= 0x0D7F for c in text)


def romanize_indic(text):
    """Romanise any Indic-script characters in `text`; Latin text passes through."""
    if not has_indic(text):
        return text
    out = []
    pending = None  # consonant waiting for its vowel
    for ch in text:
        off = _indic_offset(ch)
        if off is None:
            if pending is not None:  # word-final consonant: Hindi drops the schwa
                out.append(pending)
                pending = None
            out.append(ch)
            continue
        if off in _CONSONANTS:
            if pending is not None:
                out.append(pending + "a")
            pending = _CONSONANTS[off]
        elif off in _SIGNS:
            out.append((pending or "") + _SIGNS[off])
            pending = None
        elif off == _VIRAMA:
            if pending is not None:
                out.append(pending)
                pending = None
        elif off == _NUKTA:
            if pending is not None:
                pending = _NUKTA_MAP.get(pending, pending)
        elif off in _VOWELS:
            if pending is not None:
                out.append(pending + "a")
                pending = None
            out.append(_VOWELS[off])
        elif off in _MODIFIERS:
            if pending is not None:
                out.append(pending + "a")
                pending = None
            out.append(_MODIFIERS[off])
        elif 0x66 <= off <= 0x6F:  # native digits
            if pending is not None:
                out.append(pending + "a")
                pending = None
            out.append(str(off - 0x66))
        # anything else (danda, accents, zero-width joiners) is dropped
    if pending is not None:
        out.append(pending)
    # anusvara: "m" before a labial or at word end ("kampani", "om"), else "n"
    s = re.sub("\x02(?=[pbm]|$|[^a-z])", "m", "".join(out))
    return s.replace("\x02", "n")


_ZW_RE = re.compile("[\u200b-\u200d\u2060\ufeff]")


def to_ascii(text):
    """Romanise Indic, strip diacritics (é->e), drop anything still non-ASCII."""
    # zero-width (non-)joiners sit inside Indic words ("ಎಕ್ಸ್\u200cಪೋರ್ಟ್ಸ್"); they are not word breaks
    text = _ZW_RE.sub("", unicodedata.normalize("NFKC", text))
    text = romanize_indic(text)
    text = unicodedata.normalize("NFKD", text)
    # accents vanish (é -> e); other symbols (°, ·) become separators
    return "".join(c if ord(c) < 128 else ("" if unicodedata.combining(c) else " ") for c in text)


# ---------------------------------------------------------------------------
# Phonetic consonant skeleton (shared by English and romanised Indic)
# ---------------------------------------------------------------------------
_PHON_RULES = [
    (re.compile(r"ction"), "kshon"),  # "constructions" ~ "kanstrakshans"
    (re.compile(r"tion"), "shon"),    # "foundation" ~ "phaundeshan"
    (re.compile(r"x"), "ks"),
    (re.compile(r"ph"), "f"),
    (re.compile(r"(ck|q|c(?=[aoulrkt])|ch|kh)"), "k"),
    (re.compile(r"c"), "s"),
    (re.compile(r"g(?=[eiy])"), "j"),
    (re.compile(r"(gh|jh)"), lambda m: m.group(0)[0]),
    (re.compile(r"(th|dh|bh)"), lambda m: m.group(0)[0]),
    (re.compile(r"(sh|z)"), "s"),
    (re.compile(r"[wv]"), "b"),   # Bengali/Oriya have no v; "praibhet" ~ "private"
    (re.compile(r"d"), "t"),      # "limitet" ~ "limited", retroflex/dental merge
    (re.compile(r"ng$"), "n"),    # "injiniyarin" ~ "engineering"
]


def phonetic_key(token):
    """Consonant skeleton: first letter kept, later vowels/h/y dropped, doubles merged."""
    if not token or not token.isalpha():
        return token
    t = token
    for pat, rep in _PHON_RULES:
        t = pat.sub(rep, t)
    head = "a" if t[0] in "aeiouy" else t[0]
    tail = re.sub(r"[aeiouyh]", "", t[1:])
    s = head + tail
    return re.sub(r"(.)\1+", r"\1", s)


# ---------------------------------------------------------------------------
# Names
# ---------------------------------------------------------------------------
LEGAL_CANON = {
    "incorporated": "inc", "inc": "inc",
    "corporation": "corp", "corp": "corp",
    "company": "co", "co": "co", "cos": "co", "companies": "co",
    "limited": "ltd", "ltd": "ltd", "li": "ltd", "lim": "ltd",
    "private": "pvt", "pvt": "pvt", "pra": "pvt", "pvtltd": "pvt ltd",
    "llc": "llc", "llp": "llp", "lp": "lp", "plc": "plc", "pllc": "pllc",
    "pc": "pc", "pa": "pa", "ltda": "ltd",
    # France (test only): keep generic, no country branching
    "sarl": "sarl", "sas": "sas", "sasu": "sasu", "eurl": "eurl", "sa": "sa", "sci": "sci", "snc": "snc",
    "ets": "ets", "etablissements": "ets", "etablissement": "ets", "selarl": "selarl", "scp": "scp",
}
LEGAL_FORMS = set(LEGAL_CANON.values())
_LEGAL_SKEL = {}  # filled after phonetic_key is defined
# Honorifics / filler the noise generator prepends or appends.
NAME_FILLER = {
    "the", "ms", "smt", "shri", "sri", "mr", "mrs", "m", "s", "dr",
    "center", "centre", "partners", "service", "services", "group", "france",
}
_DOMAIN_RE = re.compile(r"^(?:https?://)?(?:www\.)?([a-z0-9][a-z0-9\-]*)\.(?:com|net|org|in|co\.in|biz|info|fr|us|co)\b")
_ID_RE = re.compile(r"\(\s*id\s*:?\s*\d+\s*\)")
_LEET = str.maketrans({"0": "o", "1": "l", "3": "e", "4": "a", "5": "s", "8": "b"})
# English and French ordinals ("3rd", "3eme"/"3ème", "1er", "2e") keep their digits;
# without the French forms the leet fixer turned "31eme" into "eleme".
_ORDINAL = re.compile(r"^\d+(st|nd|rd|th|e|er|re|eme|ieme)$")


for _w in ("incorporated", "corporation", "company", "limited", "private"):
    _LEGAL_SKEL[phonetic_key(_w)] = LEGAL_CANON[_w]


def _merge_single_letters(tokens):
    """['d','b','a'] -> ['dba'];  ['p','c'] -> ['pc'];  keeps lone letters."""
    out, run = [], []
    for t in tokens:
        if len(t) == 1 and t.isalpha():
            run.append(t)
            continue
        if run:
            out.append("".join(run))
            run = []
        out.append(t)
    if run:
        out.append("".join(run))
    return out


def _fix_leet(tok):
    if tok.isalpha() or tok.isdigit() or _ORDINAL.match(tok):
        return tok
    if sum(c.isalpha() for c in tok) >= 2:
        return tok.translate(_LEET)
    return tok


def normalize_name(raw):
    """Return dict with name tokens, core tokens, domain string and flags."""
    if raw is None or raw != raw:
        raw = ""
    s = to_ascii(str(raw)).lower().strip()
    s = _ID_RE.sub(" ", s)
    domain = ""
    m = _DOMAIN_RE.match(s.lstrip("@-. "))
    if m:
        domain = m.group(1).replace("-", "")
        s = domain
    is_handle = s.lstrip("-. ").startswith("@")
    s = s.replace("&", " and ").replace("+", " plus ")
    s = re.sub(r"[^a-z0-9]+", " ", s)
    toks = [_fix_leet(t) for t in s.split()]
    toks = [t for t in toks if t and not (t.isdigit() and len(t) >= 7)]
    toks = _merge_single_letters(toks)
    canon = []
    for t in toks:
        c = LEGAL_CANON.get(t) or (_LEGAL_SKEL.get(phonetic_key(t)) if len(t) >= 5 else None) or t
        canon.extend(c.split())
    # DBA: "X dba Y" -> core from Y, all tokens kept
    core_src = canon
    if "dba" in canon:
        i = canon.index("dba")
        core_src = canon[i + 1:] or canon[:i]
        canon = [t for t in canon if t != "dba"]
    core = [t for t in core_src if t not in LEGAL_FORMS and t not in NAME_FILLER] or \
           [t for t in core_src if t not in LEGAL_FORMS] or core_src
    return {
        "name": " ".join(canon),
        "core": " ".join(core),
        "domain": domain or ("".join(core) if is_handle else ""),
        "legal": " ".join(sorted({t for t in canon if t in LEGAL_FORMS})),
    }


# ---------------------------------------------------------------------------
# Addresses
# ---------------------------------------------------------------------------
US_STATES = {
    "al": "alabama", "ak": "alaska", "az": "arizona", "ar": "arkansas", "ca": "california",
    "co": "colorado", "ct": "connecticut", "de": "delaware", "fl": "florida", "ga": "georgia",
    "hi": "hawaii", "id": "idaho", "il": "illinois", "in": "indiana", "ia": "iowa", "ks": "kansas",
    "ky": "kentucky", "la": "louisiana", "me": "maine", "md": "maryland", "ma": "massachusetts",
    "mi": "michigan", "mn": "minnesota", "ms": "mississippi", "mo": "missouri", "mt": "montana",
    "ne": "nebraska", "nv": "nevada", "nh": "new hampshire", "nj": "new jersey", "nm": "new mexico",
    "ny": "new york", "nc": "north carolina", "nd": "north dakota", "oh": "ohio", "ok": "oklahoma",
    "or": "oregon", "pa": "pennsylvania", "ri": "rhode island", "sc": "south carolina",
    "sd": "south dakota", "tn": "tennessee", "tx": "texas", "ut": "utah", "vt": "vermont",
    "va": "virginia", "wa": "washington", "wv": "west virginia", "wi": "wisconsin", "wy": "wyoming",
    "dc": "district of columbia", "pr": "puerto rico",
}
IN_STATES = {
    "ap": ["andhra pradesh"], "ar": ["arunachal pradesh"], "as": ["assam"], "br": ["bihar"],
    "cg": ["chhattisgarh", "chattisgarh", "ct"], "ga": ["goa"], "gj": ["gujarat"],
    "hr": ["haryana"], "hp": ["himachal pradesh"], "jh": ["jharkhand"], "ka": ["karnataka"],
    "kl": ["kerala", "keralam", "keralan"], "mp": ["madhya pradesh"], "mh": ["maharashtra"], "mn": ["manipur"],
    "ml": ["meghalaya"], "mz": ["mizoram"], "nl": ["nagaland"], "od": ["odisha", "orissa", "or"],
    "pb": ["punjab", "panjab", "pajab"], "rj": ["rajasthan"], "sk": ["sikkim"], "tn": ["tamil nadu", "tamilnadu", "tamilnatu"],
    "tg": ["telangana", "ts"], "tr": ["tripura"], "up": ["uttar pradesh"], "uk": ["uttarakhand", "ut"],
    "wb": ["west bengal", "pashchim banga", "paschimbanga", "pashchimabang", "pashchimbang"], "dl": ["delhi", "dilli", "new delhi"],
    "jk": ["jammu and kashmir", "jammu kashmir"], "la": ["ladakh"], "ch": ["chandigarh"],
    "py": ["puducherry", "pondicherry"], "an": ["andaman and nicobar islands"],
    "dn": ["dadra and nagar haveli and daman and diu"], "ld": ["lakshadweep"],
}
# France is test-only: regions and départements map to one region code, since
# sources swap "Gironde" <-> "Nouvelle-Aquitaine", "Nord" <-> "Hauts-de-France".
FR_REGIONS = {
    "naq": ["nouvelle aquitaine", "gironde", "landes", "dordogne", "pyrenees atlantiques",
            "lot et garonne", "charente", "charente maritime", "vienne", "haute vienne",
            "deux sevres", "correze", "creuse"],
    "hdf": ["hauts de france", "nord", "pas de calais", "somme", "oise", "aisne"],
    "pdl": ["pays de la loire", "loire atlantique", "maine et loire", "vendee", "sarthe", "mayenne"],
    "idf": ["ile de france", "paris", "seine saint denis", "hauts de seine", "val de marne",
            "essonne", "yvelines", "val d oise", "seine et marne"],
    "ara": ["auvergne rhone alpes", "rhone", "isere", "haute savoie", "savoie", "loire", "ain", "puy de dome"],
    "occ": ["occitanie", "haute garonne", "herault", "gard", "pyrenees orientales", "aude", "tarn"],
    "pac": ["provence alpes cote d azur", "bouches du rhone", "alpes maritimes", "var", "vaucluse"],
    "ges": ["grand est", "bas rhin", "haut rhin", "moselle", "meurthe et moselle", "marne"],
    "bre": ["bretagne", "ille et vilaine", "finistere", "morbihan", "cotes d armor"],
    "nor": ["normandie", "seine maritime", "calvados", "manche", "eure", "orne"],
    "bfc": ["bourgogne franche comte", "cote d or", "doubs", "saone et loire"],
    "cvl": ["centre val de loire", "loiret", "indre et loire", "cher", "eure et loir"],
    "cor": ["corse", "corse du sud", "haute corse"],
}
STREET_CANON = {
    "street": "st", "str": "st", "saint": "st", "road": "rd", "avenue": "ave", "av": "ave",
    "drive": "dr", "lane": "ln", "boulevard": "blvd", "bd": "blvd", "bld": "blvd",
    "court": "ct", "place": "pl", "circle": "cir", "highway": "hwy", "parkway": "pkwy",
    "terrace": "terr", "trail": "trl", "square": "sq", "north": "n", "south": "s", "east": "e",
    "west": "w", "fort": "ft", "mount": "mt", "route": "rte", "nagar": "ngr",
    "opposite": "opp", "near": "nr", "sector": "sec", "building": "bldg", "floor": "fl",
    "sainte": "st", "ste": "st", "r": "rue", "imp": "impasse", "blvd": "blvd",
    "rte": "rte", "all": "allee", "che": "chemin", "chem": "chemin", "fbg": "faubourg",
    "crs": "cours", "pl": "pl", "sq": "sq", "av": "ave", "avenue": "ave",
}
# Tokens that carry no identity (unit markers, filler); numbers after them are kept.
ADDR_STOP = {
    "no", "n", "h", "hno", "house", "door", "plot", "flat", "unit", "apt", "apartment",
    "suite", "bis", "ter", "de", "la", "le", "les", "des", "du", "l", "office", "shop", "cdp", "township", "city", "null", "none", "nan", "c", "o",
    "co", "w", "s", "d", "and", "of", "the", "at", "po", "dist", "district", "taluk", "tehsil",
}
_PMB_RE = re.compile(r"\b(pmb|po box|p o box|box)\s*#?\s*\d+\b")


def _build_state_lookup():
    lut = {}
    for code, full in US_STATES.items():
        lut[full] = "us_" + code
    for code, names in IN_STATES.items():
        for n in names:
            lut[n] = "in_" + code
    for code, names in FR_REGIONS.items():
        for n in names:
            lut.setdefault(n, "fr_" + code)
    # skeleton lookup (India only) catches romanised native-script spellings ("maharashtr")
    skel = {}
    for full, code in lut.items():
        if code.startswith("in_") and len(full) >= 4:
            skel.setdefault(" ".join(phonetic_key(w) for w in full.split()), code)
    return lut, skel


_STATE_FULL, _STATE_SKEL = _build_state_lookup()


def normalize_address(raw, country=""):
    """Return dict: canonical token string, numeric tokens, first number, state code."""
    if raw is None or raw != raw:
        raw = ""
    s = to_ascii(str(raw)).lower()
    if s.strip() in {"", "none", "null", "nan", "<null>", "n/a"}:
        return {"addr": "", "nums": "", "house": "", "state": "", "pmb": 0}
    pmb = 1 if _PMB_RE.search(s) else 0
    s = _PMB_RE.sub(" ", s)
    s = re.sub(r"\bwww\.\S+", " ", s)
    # State = a whole comma-separated component ("..., TX" / "..., karnataka").
    # A 2-letter code beats a full name, so "Washington, DC" -> DC.
    ctry = (country or "").lower()
    codes = US_STATES if ctry.startswith("us") else IN_STATES if ctry.startswith("ind") else {}
    prefix = "us_" if ctry.startswith("us") else "in_"
    state, by_code, kept = "", False, []
    for comp in s.split(","):
        c = " ".join(re.sub(r"[^a-z]+", " ", comp).split())
        code = None
        if len(c) == 2 and c in codes:
            code, is_code = prefix + c, True
        elif c:
            code = _STATE_FULL.get(c) or _STATE_SKEL.get(" ".join(phonetic_key(w) for w in c.split()))
            is_code = False
            if code and re.search(r"\d", comp):  # "12 Texas Ave" is a street, not a state
                code = None
        if code and (not state or (is_code and not by_code)):
            if state and not by_code:
                kept.append(prev_comp)  # demote the earlier full name back to address text
            state, by_code, prev_comp = code, is_code, comp
            continue
        kept.append(comp)
    s = re.sub(r"[^a-z0-9]+", " ", " ".join(kept))
    s = re.sub(r"\b0+(\d)", r"\1", s)
    out = []
    for t in s.split():
        t = STREET_CANON.get(t, t)
        if t in ADDR_STOP:
            continue
        out.append(t)
    nums = [t for t in out if t.isdigit()]
    return {
        "addr": " ".join(out),
        "nums": " ".join(dict.fromkeys(nums)),
        "house": nums[0] if nums else "",
        "state": state,
        "pmb": pmb,
    }


def normalize_record(entity_id, name, address, country):
    n = normalize_name(name)
    a = normalize_address(address, country)
    ctry = re.sub(r"[^a-z]", "", to_ascii(str(country or "")).lower())
    return (
        entity_id, ctry, n["name"], n["core"], n["domain"], n["legal"],
        " ".join(phonetic_key(t) for t in n["core"].split()),
        a["addr"], a["nums"], a["house"], a["state"], a["pmb"],
        int(has_indic(str(name or ""))),
    )


RECORD_COLUMNS = [
    "entity_id", "country", "name", "core", "domain", "legal", "core_phon",
    "addr", "nums", "house", "state", "pmb", "name_indic",
]

In [ ]:
%%writefile "{SRC_DIR}/ber/prep.py"
"""Load the raw TSVs and write normalised Parquet tables (one per split/source)."""
import os
import time
from multiprocessing import Pool
from pathlib import Path

import polars as pl

from .normalize import RECORD_COLUMNS, normalize_record

# Bump whenever normalize.py changes output, so cached tables are never reused stale.
NORM_VERSION = 2


def read_source_tsv(path):
    # README format: TAB separated, header row, standard CSV quoting ("" escapes).
    return pl.read_csv(
        path, separator="\t", quote_char='"', infer_schema=False,
        missing_utf8_is_empty_string=False,
    )


def _norm_chunk(rows):
    return [normalize_record(*r) for r in rows]


def normalize_frame(df, workers=None, chunk=20_000):
    """Normalise a raw source frame -> polars frame with RECORD_COLUMNS + int row id."""
    rows = list(zip(df["entity_id"].to_list(), df["business_name"].to_list(),
                    df["business_address"].to_list(), df["country"].to_list()))
    parts = [rows[i:i + chunk] for i in range(0, len(rows), chunk)]
    workers = workers or os.cpu_count()
    if workers > 1 and len(parts) > 1:
        with Pool(workers) as pool:
            out = pool.map(_norm_chunk, parts, chunksize=1)
    else:
        out = [_norm_chunk(p) for p in parts]
    recs = [r for part in out for r in part]
    schema = {c: pl.Utf8 for c in RECORD_COLUMNS}
    schema["pmb"] = pl.Int8
    schema["name_indic"] = pl.Int8
    norm = pl.DataFrame(recs, schema=schema, orient="row")
    return norm.with_columns(
        pl.Series("raw_name", df["business_name"].fill_null("")),
        pl.Series("raw_addr", df["business_address"].fill_null("")),
        pl.Series("country_raw", df["country"].fill_null("")),
    )


def prepare(data_dir, work_dir, split, sources=("source1", "source2", "source3"), workers=None):
    """TSV -> normalised parquet. Returns dict source -> parquet path. Skips if cached."""
    work_dir = Path(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)
    out = {}
    for src in sources:
        dst = work_dir / f"{split}_{src}_norm_v{NORM_VERSION}.parquet"
        if not dst.exists():
            t = time.time()
            raw = read_source_tsv(Path(data_dir) / split / f"{split}_{src}.tsv")
            # 1M-row slices keep the Python-object working set small
            parts = [normalize_frame(raw.slice(i, 1_000_000), workers)
                     for i in range(0, len(raw), 1_000_000)]
            pl.concat(parts).write_parquet(dst)
            print(f"  normalised {split}_{src}: {len(raw):,} rows in {time.time() - t:.0f}s")
            del raw, parts
        out[src] = dst
    return out


def load_ground_truth(path):
    """Ground truth TSV -> polars frame of (s1, s23) positive pairs + list of all S1 ids."""
    gt = pl.read_csv(path, separator="\t", infer_schema=False)
    pairs = (
        gt.filter(pl.col("matched_entity_ids").is_not_null() & (pl.col("matched_entity_ids") != ""))
        .with_columns(pl.col("matched_entity_ids").str.split(","))
        .explode("matched_entity_ids")
        .rename({"source1_entity_id": "s1", "matched_entity_ids": "s23"})
        .with_columns(pl.col("s23").str.strip_chars())
    )
    return pairs, gt["source1_entity_id"]

In [ ]:
%%writefile "{SRC_DIR}/ber/geo.py"
"""Fill in missing states/regions on S2/S3 records from address tokens, using a
token -> state table learned from S1 of the same split.

S1 always carries a state; S2/S3 often drop it (35% of French S2/S3 records name
only the city). Blocking partitions by (country, state), so a missing state sends
the query to a whole-country search where the true S1 is easily cut. Only tokens
that point to one state almost exclusively are used, and a state is assigned only
when the record's tokens agree, so a wrong partition is rare (see experiments.md).
No external data: the table comes from the challenge files themselves.
"""
import polars as pl

MIN_COUNT = 20      # a token must appear in at least this many S1 addresses
MIN_PURITY = 0.97   # ... and point to one state in at least this share of them
MIN_MARGIN = 2      # winning state needs this many more votes than the runner-up


def _tokens(df):
    return (df.select("row", "country", pl.col("addr").str.split(" ").alias("t")).explode("t")
            .filter(pl.col("t").str.len_chars() >= 4, ~pl.col("t").str.contains(r"\d")).unique())


def state_table(s1):
    """(country, token) -> state for tokens that locate a state reliably."""
    t = _tokens(s1.select("country", "addr", "state").with_row_index("row")) \
        .join(s1.select("state").with_row_index("row"), on="row")
    c = t.group_by("country", "t", "state").len("n")
    tot = c.group_by("country", "t").agg(pl.col("n").sum().alias("tot"))
    best = c.sort("n", descending=True).group_by("country", "t").first().join(tot, on=["country", "t"])
    return best.filter((pl.col("tot") >= MIN_COUNT) & (pl.col("n") / pl.col("tot") >= MIN_PURITY)) \
               .select("country", "t", pl.col("state").alias("inf_state"))


def infer_states(s23, table):
    """Return s23 with empty `state` filled where its address tokens agree on one state.
    Adds `state_inferred` (1 where a state was filled)."""
    need = s23.select("country", "addr", "state").with_row_index("row").filter(pl.col("state") == "")
    votes = _tokens(need).join(table, on=["country", "t"]).group_by("row", "inf_state").len("v")
    ranked = votes.sort(["row", "v"], descending=[False, True]).group_by("row", maintain_order=True).agg(
        pl.col("inf_state").first(), pl.col("v").first().alias("v1"), pl.col("v").get(1, null_on_oob=True).fill_null(0).alias("v2"))
    ok = ranked.filter(pl.col("v1") - pl.col("v2") >= MIN_MARGIN - 1).filter(pl.col("v1") > pl.col("v2"))
    fill = pl.DataFrame({"row": pl.Series(range(len(s23)), dtype=pl.UInt32)}).join(
        ok.select("row", "inf_state"), on="row", how="left")["inf_state"]
    return s23.with_columns(
        pl.when((pl.col("state") == "") & fill.is_not_null()).then(fill).otherwise(pl.col("state")).alias("state"),
        ((pl.col("state") == "") & fill.is_not_null()).cast(pl.Int8).alias("state_inferred"),
    )

In [ ]:
%%writefile "{SRC_DIR}/ber/block.py"
"""Candidate generation: sparse IDF token retrieval from S2/S3 records to S1.

Why this direction: in the training labels every S2/S3 record matches at most
one S1 record, so each S2/S3 record only needs its best few S1 neighbours.

Two channels, both cosine similarity of binary IDF-weighted token vectors:
- "full": name tokens + phonetic name keys + address tokens (+ house/street key).
  Searches the query's (country, state); a query with no state searches its country.
- "name": name tokens only, for queries whose address is missing or too short
  to carry evidence (a name-only query scored against a name+address vector
  would otherwise get a tiny cosine).
Scores within a partition use that partition's IDF. Top-n per query is done
inside the sparse product (sparse_dot_topn, Apache-2.0), so the full score
matrix is never materialised.
"""
import time

import numpy as np
import polars as pl
import scipy.sparse as sp
from sparse_dot_topn import sp_matmul_topn

# Parameters chosen by a recall-vs-volume sweep on train (experiments.md, 2026-09-25):
# address-less queries recall 75.7% -> 84.9% with the wider name channel; Indian
# states 96.5% -> 97.0% with top-8/0.5; US states already 99.7% at top-6/0.6.
CFG = {
    "max_df": 20_000,     # tokens more common than this within a partition are ignored
    "top_k": 8,           # S1 neighbours kept per S2/S3 record (full channel)
    "min_score": 0.08,    # cosine floor
    "min_rel": 0.5,       # keep neighbours scoring >= min_rel * best score of that query
    "name_top_k": 25,     # name channel for queries with NO address: the name is all there is
    "name_min_rel": 0.3,
    "short_top_k": 4,     # name channel for queries with 1..short_addr address tokens
    "short_min_rel": 0.7,
    "short_addr": 2,
    "query_chunk": 200_000,
}
# Per-country overrides of the full channel. Countries not listed (e.g. France,
# unseen in train) keep the wider defaults above, which favour recall.
COUNTRY_CFG = {"us": {"top_k": 6, "min_rel": 0.6}}


# States the sources disagree on (Hyderabad is filed under both TG and AP).
STATE_NEIGHBOURS = {"in_tg": ["in_ap"], "in_ap": ["in_tg"]}
NAME_PREFIXES = ("n", "p", "d")


def token_frame(df):
    """Explode a normalised frame into (row, token); the first char marks the field."""
    def split(col, prefix, min_len=2):
        t = df.select("row", pl.col(col).str.split(" ").alias("t")).explode("t")
        return t.filter(pl.col("t").str.len_chars() >= min_len).with_columns((prefix + pl.col("t")).alias("t"))
    name, phon = split("core", "n"), split("core_phon", "p")
    addr = df.select("row", pl.col("addr").str.split(" ").alias("t")).explode("t")
    addr = addr.filter((pl.col("t").str.len_chars() >= 3) | pl.col("t").str.contains(r"^\d+$"))
    addr = addr.with_columns(("a" + pl.col("t")).alias("t"))
    # house number + first street word: rare, precise key
    hs = df.filter(pl.col("house") != "").select(
        "row", ("h" + pl.col("house") + "_" + pl.col("addr").str.extract(r"\d+\s+([a-z]{3,})", 1)).alias("t"),
    ).drop_nulls()
    # whole name without spaces: meets domains/handles ("nexosidea.com" ~ "Nexos Idea")
    flat = pl.when(pl.col("domain") != "").then(pl.col("domain")).otherwise(pl.col("core").str.replace_all(" ", ""))
    dom = df.select("row", ("d" + flat).alias("t")).filter(pl.col("t").str.len_chars() >= 5)
    return pl.concat([name, phon, addr, hs, dom]).unique()


def _csr(rows, cols, vals, n_rows, n_cols):
    m = sp.csr_matrix((vals, (rows, cols)), shape=(n_rows, n_cols), dtype=np.float32)
    norms = np.sqrt(np.asarray(m.multiply(m).sum(axis=1)).ravel())
    norms[norms == 0] = 1.0
    return sp.diags(1.0 / norms).dot(m).tocsr()


def _gather(ptr, rows):
    """Flat indices of the CSR-style segments ptr[r]:ptr[r+1] for all r in rows."""
    lens = ptr[rows + 1] - ptr[rows]
    offs = np.repeat(ptr[rows] - np.concatenate([[0], np.cumsum(lens)[:-1]]), lens)
    return np.arange(lens.sum()) + offs, lens


class _Index:
    """Token postings for one country in CSR-like form (local row ids)."""

    def __init__(self, tok, n_rows, keep_mask=None):
        if keep_mask is not None:
            tok = tok.filter(keep_mask)
        tok = tok.sort("lr")
        self.tids = tok["tid"].to_numpy()
        self.ptr = np.searchsorted(tok["lr"].to_numpy(), np.arange(n_rows + 1))


def _search(d_idx, q_idx, d_rows, q_rows, n_vocab, top_k, min_rel, n_threads):
    """Cosine top-k of q_rows against d_rows. Returns local (q, d, score, rank) arrays."""
    idx, lens = _gather(d_idx.ptr, d_rows)
    d_tids = d_idx.tids[idx]
    df_counts = np.bincount(d_tids, minlength=n_vocab)
    idf = np.log1p(len(d_rows) / np.maximum(df_counts, 1)).astype(np.float32)
    ok = df_counts[d_tids] <= CFG["max_df"]
    d_local = np.repeat(np.arange(len(d_rows)), lens)
    DT = _csr(d_local[ok], d_tids[ok], idf[d_tids[ok]], len(d_rows), n_vocab).T.tocsr()
    res = []
    for q0 in range(0, len(q_rows), CFG["query_chunk"]):
        qr = q_rows[q0:q0 + CFG["query_chunk"]]
        qidx, qlens = _gather(q_idx.ptr, qr)
        q_tids = q_idx.tids[qidx]
        # queries share the partition idf; tokens unseen in S1 carry no evidence
        w = idf[q_tids] * ((df_counts[q_tids] > 0) & (df_counts[q_tids] <= CFG["max_df"]))
        Q = _csr(np.repeat(np.arange(len(qr)), qlens), q_tids, w, len(qr), n_vocab)
        C = sp_matmul_topn(Q, DT, top_n=top_k, threshold=CFG["min_score"], sort=True, n_threads=n_threads)
        counts = np.diff(C.indptr)
        r = np.repeat(np.arange(len(qr)), counts)
        first = np.repeat(C.indptr[:-1], counts)
        rank = np.arange(len(r)) - first
        keep = C.data >= min_rel * C.data[first]
        res.append((qr[r[keep]], d_rows[C.indices[keep]], C.data[keep], rank[keep]))
    return [np.concatenate(x) for x in zip(*res)]


def _block_country(s1, s23, n_threads, verbose, t0, top_k, min_rel):
    tok1 = token_frame(s1.select("row", "core", "core_phon", "addr", "house", "domain"))
    tok2 = token_frame(s23.select("row", "core", "core_phon", "addr", "house", "domain"))
    vocab = pl.concat([tok1["t"], tok2["t"]]).unique().to_frame("t").with_row_index("tid")
    n_vocab = len(vocab)
    tok1 = tok1.join(vocab, on="t").join(s1.select("row").with_row_index("lr"), on="row")
    tok2 = tok2.join(vocab, on="t").join(s23.select("row").with_row_index("lr"), on="row")
    del vocab
    is_name = pl.col("t").str.slice(0, 1).is_in(NAME_PREFIXES)
    full1, full2 = _Index(tok1, len(s1)), _Index(tok2, len(s23))
    name1, name2 = _Index(tok1, len(s1), is_name), _Index(tok2, len(s23), is_name)
    del tok1, tok2
    st1, st2 = s1["state"].to_numpy(), s23["state"].to_numpy()
    n_addr = s23["addr"].str.split(" ").list.len().to_numpy() * (s23["addr"].to_numpy() != "")
    g1, g2 = s1["row"].to_numpy(), s23["row"].to_numpy()
    out = []
    for st in np.unique(st2):
        q_rows = np.flatnonzero(st2 == st)
        # search space: same (or neighbouring) state plus stateless S1; no state -> whole country
        d_rows = np.arange(len(s1)) if st == "" else \
            np.flatnonzero(np.isin(st1, [st, ""] + STATE_NEIGHBOURS.get(st, [])))
        if len(d_rows) == 0:
            continue
        empty = q_rows[n_addr[q_rows] == 0]
        short = q_rows[(n_addr[q_rows] > 0) & (n_addr[q_rows] <= CFG["short_addr"])]
        for chan, di, qi, qr, k, rel in [
            ("full", full1, full2, q_rows, top_k, min_rel),
            ("name", name1, name2, empty, CFG["name_top_k"], CFG["name_min_rel"]),
            ("name", name1, name2, short, CFG["short_top_k"], CFG["short_min_rel"]),
        ]:
            if len(qr) == 0:
                continue
            q, d, v, rk = _search(di, qi, d_rows, qr, n_vocab, k, rel, n_threads)
            out.append((chan, pl.DataFrame({
                "s23_row": g2[q].astype(np.int32), "s1_row": g1[d].astype(np.int32),
                f"blk_{chan}": v.astype(np.float32), f"blk_{chan}_rank": rk.astype(np.int8),
            })))
        if verbose and len(q_rows) > 300_000:
            print(f"    {st or '<no state>'}: S1 {len(d_rows):,} x S23 {len(q_rows):,}  ({time.time() - t0:.0f}s)")
    return out


def generate_candidates(s1, s23, n_threads=-1, verbose=True):
    """s1, s23: normalised frames. Returns one row per candidate pair:
    s23_row, s1_row (row numbers in the inputs), blk_full, blk_full_rank, blk_name, blk_name_rank.
    Country is an open set: every country (including unseen ones) is blocked on its own."""
    t0 = time.time()
    cols = ["country", "state", "core", "core_phon", "addr", "house", "domain"]
    s1 = s1.select(cols).with_row_index("row")
    s23 = s23.select(cols).with_row_index("row")
    out = []
    for ctry in sorted(set(s23["country"].unique().to_list())):
        a, b = s1.filter(pl.col("country") == ctry), s23.filter(pl.col("country") == ctry)
        if len(a) == 0:
            continue
        if verbose:
            print(f"  country {ctry!r}: S1 {len(a):,}  S23 {len(b):,}")
        cc = {**CFG, **COUNTRY_CFG.get(ctry, {})}
        out += _block_country(a, b, n_threads, verbose, t0, cc["top_k"], cc["min_rel"])
    cands = pl.concat([df for c, df in out if c == "full"])
    name = [df for c, df in out if c == "name"]
    if name:
        cands = cands.join(pl.concat(name), on=["s23_row", "s1_row"], how="full", coalesce=True)
    cands = cands.with_columns(
        pl.col("blk_full").fill_null(0.0), pl.col("blk_full_rank").fill_null(99),
        pl.col("blk_name").fill_null(0.0), pl.col("blk_name_rank").fill_null(99),
    )
    if verbose:
        print(f"  candidates: {len(cands):,} pairs, {len(cands) / max(len(s23), 1):.2f} per S2/S3 "
              f"({time.time() - t0:.0f}s)")
    return cands

In [ ]:
%%writefile "{SRC_DIR}/ber/features.py"
"""Pairwise features for (S1, S2/S3) candidate pairs.

String similarities use rapidfuzz.process.cpdist (element-wise, multithreaded).
Country is deliberately NOT a feature: France is unseen in training, so the
model must rely on country-agnostic evidence. Missing fields become NaN, which
XGBoost treats as "no evidence" rather than disagreement.
"""
import math

import numpy as np
import polars as pl
from rapidfuzz import fuzz, process
from rapidfuzz.distance import JaroWinkler, Levenshtein

TEXT_COLS = ["name", "core", "core_phon", "domain", "legal", "addr", "nums", "house", "state", "pmb",
             "name_indic", "core_freq", "core_n23", "phon_freq", "phon_n23"]


def record_stats(s1, s23):
    """Name-rarity statistics, measured on the tables of the same split.

    Per record (both frames), keyed on its exact core name and on its phonetic key:
    - core_freq / phon_freq: how many S1 records carry it
    - core_n23 / phon_n23:  how many S2/S3 records carry it
    Every S1 entity has ~3.5 S2/S3 records, so many more S2/S3 than S1 records with a
    name means other businesses ("orphans" absent from S1) share it; that is the main
    source of false merges on address-less records. Also returns an IDF table over
    phonetic name tokens.
    """
    for key, f1, f23 in [("core", "core_freq", "core_n23"), ("core_phon", "phon_freq", "phon_n23")]:
        c1 = s1.group_by(key).len(f1)
        c23 = s23.group_by(key).len(f23)
        s1 = s1.join(c1, on=key, how="left", maintain_order="left").join(c23, on=key, how="left", maintain_order="left")
        s23 = s23.join(c1, on=key, how="left", maintain_order="left").join(c23, on=key, how="left", maintain_order="left")
        s1 = s1.with_columns(pl.col(f1).fill_null(0), pl.col(f23).fill_null(0))
        s23 = s23.with_columns(pl.col(f1).fill_null(0), pl.col(f23).fill_null(0))
    tok = s1.select(pl.col("core_phon").str.split(" ").list.unique().alias("t")).explode("t")
    # pl.lit(python float) + explicit cast before .log(): a bare numpy scalar (np.log(...))
    # combined with a polars Expr via an operator has caused an engine-side panic
    # (dtype coercion of the numpy scalar into the query, version-dependent) in the past.
    log_n = pl.lit(math.log(len(s1)), dtype=pl.Float64)
    idf = tok.filter(pl.col("t") != "").group_by("t").len("df").with_columns(
        (log_n - pl.col("df").cast(pl.Float64).log()).cast(pl.Float32).alias("w")).select("t", "w")
    return s1, s23, idf


def _idf_overlap(a, b, idf, default_w):
    """IDF mass of shared / union tokens and the rarest shared token."""
    df = pl.DataFrame({"a": a, "b": b}).with_row_index("i").with_columns(
        pl.col("a").str.split(" ").list.unique(), pl.col("b").str.split(" ").list.unique())
    def mass(col):
        e = df.select("i", col).explode(col).rename({col: "t"}).filter(pl.col("t") != "")
        e = e.join(idf, on="t", how="left").with_columns(pl.col("w").fill_null(default_w))
        g = e.group_by("i").agg(pl.col("w").sum().alias("s"), pl.col("w").max().alias("m"))
        g = pl.DataFrame({"i": np.arange(len(df), dtype=np.uint32)}).join(g, on="i", how="left")
        return g["s"].fill_null(0).to_numpy(), g["m"].fill_null(0).to_numpy()
    df = df.with_columns(pl.col("a").list.set_intersection("b").alias("sh"), pl.col("a").list.set_union("b").alias("un"))
    sh, sh_max = mass("sh")
    un, _ = mass("un")
    with np.errstate(divide="ignore", invalid="ignore"):
        jac = np.where(un > 0, sh / un, np.nan)
    return sh.astype(np.float32), sh_max.astype(np.float32), jac.astype(np.float32)


def _sim(scorer, a, b, **kw):
    return process.cpdist(a, b, scorer=scorer, workers=-1, dtype=np.float32, **kw)


def _set_overlap(a, b):
    """Jaccard, containment(a in b), containment(b in a) of space-separated token sets."""
    df = pl.DataFrame({"a": a, "b": b}).with_columns(
        pl.col("a").str.split(" ").list.eval(pl.element().filter(pl.element() != "")).list.unique(),
        pl.col("b").str.split(" ").list.eval(pl.element().filter(pl.element() != "")).list.unique(),
    ).with_columns(
        pl.col("a").list.set_intersection("b").list.len().alias("i"),
        pl.col("a").list.len().alias("la"), pl.col("b").list.len().alias("lb"),
    )
    i, la, lb = (df[c].to_numpy().astype(np.float32) for c in ("i", "la", "lb"))
    with np.errstate(divide="ignore", invalid="ignore"):
        jac = i / (la + lb - i)
        ca = i / la
        cb = i / lb
    return jac, ca, cb


def pair_features(pairs, s1, s23, idf):
    """pairs: frame with s1_row, s23_row + blocking columns. s1/s23: normalised frames.

    Returns a polars frame of float32 features aligned with `pairs`.
    """
    a = s1.select(TEXT_COLS)[pairs["s1_row"].to_numpy()]
    b = s23.select(TEXT_COLS)[pairs["s23_row"].to_numpy()]
    f = {}
    an, bn = a["name"].to_list(), b["name"].to_list()
    ac, bc = a["core"].to_list(), b["core"].to_list()
    ap, bp = a["core_phon"].to_list(), b["core_phon"].to_list()
    f["name_ratio"] = _sim(fuzz.ratio, an, bn)
    f["name_tset"] = _sim(fuzz.token_set_ratio, an, bn)
    f["name_tsort"] = _sim(fuzz.token_sort_ratio, an, bn)
    f["core_ratio"] = _sim(fuzz.ratio, ac, bc)
    f["core_tset"] = _sim(fuzz.token_set_ratio, ac, bc)
    f["core_partial"] = _sim(fuzz.partial_ratio, ac, bc)
    f["core_jw"] = _sim(JaroWinkler.normalized_similarity, ac, bc)
    f["core_lev"] = _sim(Levenshtein.distance, ac, bc)
    f["phon_ratio"] = _sim(fuzz.ratio, ap, bp)
    f["phon_tset"] = _sim(fuzz.token_set_ratio, ap, bp)
    f["phon_tsort"] = _sim(fuzz.token_sort_ratio, ap, bp)
    jac, ca, cb = _set_overlap(ac, bc)
    f["core_jac"], f["core_in_s23"], f["s23_in_core"] = jac, ca, cb
    jac, ca, cb = _set_overlap(ap, bp)
    f["phon_jac"], f["phon_in_s23"], f["s23_in_phon"] = jac, ca, cb
    # name rarity: IDF mass of shared phonetic tokens, and how common each exact name is
    default_w = float(idf["w"].max()) if len(idf) else 10.0
    f["idf_shared"], f["idf_shared_max"], f["idf_jac"] = _idf_overlap(ap, bp, idf, default_w)
    f["core_freq_s1"] = a["core_freq"].to_numpy().astype(np.float32)
    f["core_freq_s23"] = b["core_freq"].to_numpy().astype(np.float32)
    # orphan pressure: S2/S3 records sharing the name per S1 record sharing it (~3.5 expected)
    f["core_n23_s23"] = b["core_n23"].to_numpy().astype(np.float32)
    f["core_n23_per_s1"] = (b["core_n23"].to_numpy() / (b["core_freq"].to_numpy() + 1)).astype(np.float32)
    f["phon_freq_s23"] = b["phon_freq"].to_numpy().astype(np.float32)
    f["phon_n23_per_s1"] = (b["phon_n23"].to_numpy() / (b["phon_freq"].to_numpy() + 1)).astype(np.float32)
    f["s1_core_n23"] = a["core_n23"].to_numpy().astype(np.float32)
    f["core_eq"] = (a["core"].to_numpy() == b["core"].to_numpy()).astype(np.float32)
    # domains / handles: compare against the other side's core with spaces removed
    adom, bdom = a["domain"].to_numpy(), b["domain"].to_numpy()
    a_flat = [x.replace(" ", "") for x in ac]
    b_flat = [x.replace(" ", "") for x in bc]
    dom = _sim(fuzz.partial_ratio, a_flat, b_flat)
    has_dom = (adom != "") | (bdom != "")
    f["dom_partial"] = np.where(has_dom, dom, np.nan).astype(np.float32)
    f["flat_ratio"] = _sim(fuzz.ratio, a_flat, b_flat)
    f["s23_is_domain"] = (bdom != "").astype(np.float32)
    # legal form agreement (NaN when either side has none)
    al, bl = a["legal"].to_numpy(), b["legal"].to_numpy()
    f["legal_eq"] = np.where((al != "") & (bl != ""), (al == bl).astype(np.float32), np.nan).astype(np.float32)
    f["len_core_s1"] = np.fromiter((len(x) for x in ac), np.float32, len(ac))
    f["len_core_s23"] = np.fromiter((len(x) for x in bc), np.float32, len(bc))
    f["ntok_core_s1"] = np.fromiter((x.count(" ") + 1 if x else 0 for x in ac), np.float32, len(ac))
    f["s23_indic"] = b["name_indic"].to_numpy().astype(np.float32)

    # address
    aa, ba = a["addr"].to_list(), b["addr"].to_list()
    miss = (a["addr"].to_numpy() == "") | (b["addr"].to_numpy() == "")
    for k, scorer in [("addr_ratio", fuzz.ratio), ("addr_tset", fuzz.token_set_ratio),
                      ("addr_tsort", fuzz.token_sort_ratio), ("addr_partial", fuzz.partial_ratio)]:
        f[k] = np.where(miss, np.nan, _sim(scorer, aa, ba)).astype(np.float32)
    jac, ca, cb = _set_overlap(aa, ba)
    f["addr_jac"] = np.where(miss, np.nan, jac).astype(np.float32)
    f["addr_in_s23"] = np.where(miss, np.nan, ca).astype(np.float32)
    f["s23_in_addr"] = np.where(miss, np.nan, cb).astype(np.float32)
    f["s23_addr_missing"] = (b["addr"].to_numpy() == "").astype(np.float32)
    jac, ca, cb = _set_overlap(a["nums"].to_list(), b["nums"].to_list())
    nmiss = (a["nums"].to_numpy() == "") | (b["nums"].to_numpy() == "")
    f["nums_jac"] = np.where(nmiss, np.nan, jac).astype(np.float32)
    f["nums_s23_in_s1"] = np.where(nmiss, np.nan, cb).astype(np.float32)
    ah, bh = a["house"].to_numpy(), b["house"].to_numpy()
    f["house_eq"] = np.where((ah != "") & (bh != ""), (ah == bh).astype(np.float32), np.nan).astype(np.float32)
    f["house_lev"] = np.where((ah != "") & (bh != ""),
                              _sim(Levenshtein.distance, ah.tolist(), bh.tolist()), np.nan).astype(np.float32)
    # house numbers are perturbed arithmetically (15022 -> 15023); near-duplicate branches differ slightly too
    hn1 = pl.Series(ah).str.slice(0, 9).cast(pl.Float64, strict=False).to_numpy()
    hn2 = pl.Series(bh).str.slice(0, 9).cast(pl.Float64, strict=False).to_numpy()
    with np.errstate(invalid="ignore", divide="ignore"):
        f["house_absdiff"] = np.abs(hn1 - hn2).astype(np.float32)
        f["house_reldiff"] = (np.abs(hn1 - hn2) / np.maximum(np.maximum(hn1, hn2), 1)).astype(np.float32)
    f["nums_eq"] = np.where(nmiss, np.nan, (a["nums"].to_numpy() == b["nums"].to_numpy()).astype(np.float32)).astype(np.float32)
    ast, bst = a["state"].to_numpy(), b["state"].to_numpy()
    f["state_eq"] = np.where((ast != "") & (bst != ""), (ast == bst).astype(np.float32), np.nan).astype(np.float32)
    f["s23_pmb"] = b["pmb"].to_numpy().astype(np.float32)
    # combined evidence
    f["name_x_addr"] = f["core_tset"] * np.nan_to_num(f["addr_tset"], nan=50.0) / 100.0
    return pl.DataFrame(f)


def group_features(pairs, score_col):
    """Context features: how a pair's score compares with its competitors.

    - within the S2/S3 record's candidate list (which S1 does it belong to?)
    - within the S1 record's candidate list (how many strong matches does it have?)
    """
    s = pl.col(score_col)
    return pairs.with_columns(
        s.rank("ordinal", descending=True).over("s23_row").cast(pl.Int16).alias(f"{score_col}_rk23"),
        (s - s.max().over("s23_row")).alias(f"{score_col}_gap23"),
        # best candidate: lead over the runner-up; others: deficit to the best
        pl.when(s == s.max().over("s23_row"))
        .then(s - s.filter(s < s.max()).max().over("s23_row"))
        .otherwise(s - s.max().over("s23_row"))
        .fill_null(1.0).alias(f"{score_col}_margin23"),
        pl.len().over("s23_row").cast(pl.Int16).alias(f"{score_col}_n23"),
        s.rank("ordinal", descending=True).over("s1_row").cast(pl.Int16).alias(f"{score_col}_rk1"),
        (s - s.max().over("s1_row")).alias(f"{score_col}_gap1"),
        pl.len().over("s1_row").cast(pl.Int16).alias(f"{score_col}_n1"),
        s.mean().over("s1_row").alias(f"{score_col}_mean1"),
    )


CONTEXT_NAMES = [
    "p1", "p1_rk23", "p1_gap23", "p1_margin23", "p1_n23", "p1_rk1", "p1_gap1", "p1_n1", "p1_mean1",
    "p1_sum1", "p1_n05_1", "p1_max1_src", "p1_sum23", "p1_rk1_src",
    "sib1_p", "sib1_name", "sib1_addr", "sib1_house_eq", "sib2_p", "sib2_name", "sib2_addr",
]


def write_context(X, col0, pairs, p1, s23, chunk=2_000_000):
    """Stage-2 context features, written straight into X[:, col0:col0+21] (in place).

    Memory matters here: at full scale X is ~10 GB and this step runs on ~28M pairs.
    Building all 21 columns as one frame and converting it (float64 up-cast + float32
    copy) needed ~10 GB extra and killed the full run, so each column is computed on a
    slim 4-column frame and written immediately, and siblings use numpy indexing.

    Columns:
    - competitor context of stage-1 scores within the S2/S3 record's and the S1's
      candidate lists, expected matches per S1 and per source, rank within the source;
    - siblings (S2<->S3 triangulation): similarity of the record to the S1's two
      most confident other candidates. When an S1 truly matches several records they
      describe the same business, so a weak record that resembles a confident sibling
      is likely a match; a lookalike orphan resembles the S1 but not its cluster.
    Returns the column names (CONTEXT_NAMES).
    """
    n = len(pairs)
    s1r = pairs["s1_row"].to_numpy().astype(np.int64)
    s23r = pairs["s23_row"].to_numpy().astype(np.int64)
    base = pl.DataFrame({"s1_row": s1r, "s23_row": s23r, "is_s3": pairs["is_s3"], "p1": p1})
    s = pl.col("p1")
    exprs = [
        s,
        s.rank("ordinal", descending=True).over("s23_row"),
        s - s.max().over("s23_row"),
        # best candidate: lead over the runner-up; others: deficit to the best
        pl.when(s == s.max().over("s23_row"))
        .then(s - s.filter(s < s.max()).max().over("s23_row"))
        .otherwise(s - s.max().over("s23_row")).fill_null(1.0),
        pl.len().over("s23_row"),
        s.rank("ordinal", descending=True).over("s1_row"),
        s - s.max().over("s1_row"),
        pl.len().over("s1_row"),
        s.mean().over("s1_row"),
        s.sum().over("s1_row"),
        (s > 0.5).sum().over("s1_row"),
        s.max().over(["s1_row", "is_s3"]),
        s.sum().over("s23_row"),
        s.rank("ordinal", descending=True).over(["s1_row", "is_s3"]),
    ]
    for k, e in enumerate(exprs):
        X[:, col0 + k] = base.select(e.cast(pl.Float32)).to_series().to_numpy()
    del base

    # siblings: the S1's top-3 candidates by p1 (stable order), minus the pair itself
    order = np.lexsort((-p1, s1r))
    s1_sorted = s1r[order]
    starts = np.flatnonzero(np.r_[True, s1_sorted[1:] != s1_sorted[:-1]])
    lens = np.diff(np.r_[starts, n])
    grp = np.empty(n, dtype=np.int64)
    grp[order] = np.repeat(np.arange(len(starts)), lens)
    del s1_sorted
    me = np.arange(n)
    top = []
    for j in range(3):
        t = np.where(j < lens, order[np.minimum(starts + j, n - 1)], -1)[grp]
        top.append(t)
    del order, grp, starts, lens
    sa = np.where(top[0] == me, top[1], top[0])
    sb = np.where((top[0] == me) | (top[1] == me), top[2], top[1])
    del top, me
    txt = s23.select("core", "addr", "house")
    k = col0 + 14
    for tag, sib in [("sib1", sa), ("sib2", sb)]:
        has = sib >= 0
        cols = {"p": k, "name": k + 1, "addr": k + 2}
        if tag == "sib1":
            cols["house"] = k + 3
        for c in cols.values():
            X[:, c] = np.nan
        X[has, cols["p"]] = p1[sib[has]]
        idx = np.flatnonzero(has)
        for i in range(0, len(idx), chunk):
            j = idx[i:i + chunk]
            a = txt[s23r[j]]
            b = txt[s23r[sib[j]]]
            X[j, cols["name"]] = _sim(fuzz.token_set_ratio, a["core"].to_list(), b["core"].to_list())
            both = (a["addr"].to_numpy() != "") & (b["addr"].to_numpy() != "")
            ad = _sim(fuzz.token_set_ratio, a["addr"].to_list(), b["addr"].to_list())
            X[j, cols["addr"]] = np.where(both, ad, np.nan)
            if tag == "sib1":
                ha, hb = a["house"].to_numpy(), b["house"].to_numpy()
                X[j, cols["house"]] = np.where((ha != "") & (hb != ""), (ha == hb).astype(np.float32), np.nan)
        k += len(cols)
    return list(CONTEXT_NAMES)

In [ ]:
%%writefile "{SRC_DIR}/ber/model.py"
"""Two-stage gradient-boosted scorer (XGBoost, Apache-2.0), GPU when available.

Stage 1 scores each pair from its own features (strings + blocking).
Stage 2 adds competitor context computed from stage-1 probabilities: how a pair
ranks among the other S1 candidates of the same S2/S3 record and among the
other candidates of the same S1. Stage-1 probabilities used as stage-2 inputs
on training data are out-of-fold, so stage 2 never sees leaked scores.

Folds/variants train SEQUENTIALLY, one GPU at a time, even with several available
(Kaggle 2x T4): boolean-indexing a fold's rows out of the shared feature matrix (X[tr],
X[va]) always copies, never views, and at full scale X itself is 10+ GB, so two folds
copying concurrently can exceed available RAM (observed: OOM during stage 1 on the full
run). Each fold's copies are freed before the next fold starts. GPUs still rotate across
folds/variants via `_device`, so both GPUs get used, just not at the same instant.
"""
import gc
import subprocess
import warnings

import numpy as np
import polars as pl
import xgboost as xgb

from .features import CONTEXT_NAMES

N_CONTEXT = len(CONTEXT_NAMES)  # stage-2 context columns written by features.write_context

XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "learning_rate": 0.08,
    "max_depth": 10,
    "min_child_weight": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "lambda": 1.0,
    "max_bin": 256,
    "seed": 42,
}
PREDICT_CHUNK = 2_000_000

warnings.filterwarnings("ignore", message=".*Falling back to prediction using DMatrix.*")


def _ram():
    try:
        import psutil
        vm = psutil.virtual_memory()
        return f" | RAM {psutil.Process().memory_info().rss / 1e9:.1f} GB used by us, {vm.available / 1e9:.1f} GB free"
    except Exception:
        return ""


def n_gpus():
    try:
        out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=20).stdout
        return sum(1 for line in out.splitlines() if line.startswith("GPU "))
    except Exception:
        return 0


def _device(i=0):
    g = n_gpus()
    return f"cuda:{i % g}" if g else "cpu"


class Model:
    """Thin wrapper so the pipeline does not depend on the booster API."""

    def __init__(self, booster, names):
        self.booster, self.names = booster, names
        bi = getattr(booster, "best_iteration", None)
        self.best_iteration = int(bi) if bi is not None else booster.num_boosted_rounds() - 1

    def predict(self, X, rows=None):
        """Predict X (or only X[rows]) in chunks, so no large copy of X is ever made."""
        rng = (0, self.best_iteration + 1)
        n = len(X) if rows is None else len(rows)
        out = np.empty(n, dtype=np.float32)
        for i in range(0, n, PREDICT_CHUNK):
            part = X[i:i + PREDICT_CHUNK] if rows is None else X[rows[i:i + PREDICT_CHUNK]]
            out[i:i + PREDICT_CHUNK] = self.booster.predict(
                xgb.DMatrix(part, feature_names=self.names), iteration_range=rng)
        return out

    def importance(self):
        g = self.booster.get_score(importance_type="total_gain")
        return sorted(g.items(), key=lambda t: -t[1])

    def save(self, path):
        self.booster.save_model(str(path))


def train_one(X, y, Xv, yv, rounds, names, device=None, params=None):
    device = device or _device(0)
    p = {**XGB_PARAMS, **(params or {}), "device": device}
    dtr = xgb.QuantileDMatrix(X, y, feature_names=names, max_bin=p["max_bin"])
    dva = xgb.QuantileDMatrix(Xv, yv, ref=dtr, feature_names=names)
    b = xgb.train(p, dtr, num_boost_round=rounds, evals=[(dva, "valid")],
                  early_stopping_rounds=50, verbose_eval=200)
    del dtr, dva
    return Model(b, names)


def stage1_oof(X, y, s1_row, fit_mask, names, rounds=2000, n_folds=3, seed=42):
    """Out-of-fold stage-1 probabilities for fit rows; fold-average for all other rows.
    Folds are grouped by S1 entity. Folds run concurrently across available GPUs."""
    rng = np.random.default_rng(seed)
    ents = np.unique(s1_row[fit_mask])
    fold_df = pl.DataFrame({"r": ents, "f": rng.integers(0, n_folds, len(ents))})
    fold = pl.DataFrame({"r": s1_row}).join(fold_df, on="r", how="left", maintain_order="left")["f"] \
        .fill_null(-1).to_numpy()
    fold = np.where(fit_mask, fold, -1)
    p = np.zeros(len(y), dtype=np.float32)
    other = fold == -1
    models = []
    for f in range(n_folds):
        tr, va = fit_mask & (fold != f), fold == f
        Xtr, Xva = X[tr], X[va]  # copies (boolean index); freed at the end of this iteration
        m = train_one(Xtr, y[tr], Xva, y[va], rounds, names, device=_device(f))
        del Xtr, Xva
        print(f"    stage1 fold {f}: best_iter {m.best_iteration}{_ram()}", flush=True)
        p[va] = m.predict(X, rows=np.flatnonzero(va))
        if other.any():
            p[other] += m.predict(X, rows=np.flatnonzero(other)) / n_folds
        models.append(m)
        gc.collect()
        print(f"    stage1 fold {f}: out-of-fold scores done{_ram()}", flush=True)
    return p, models


def stage1_predict(models, X):
    return np.mean([m.predict(X) for m in models], axis=0).astype(np.float32)


# Stage 2 is an average of two differently-shaped boosters (depth-wise deep vs
# shallower + stronger column sampling). Trained sequentially (see module docstring).
STAGE2_VARIANTS = [
    {},
    {"max_depth": 7, "colsample_bytree": 0.6, "subsample": 0.9, "min_child_weight": 10, "seed": 7},
]


class Ensemble:
    def __init__(self, models):
        self.models = models
        self.best_iteration = [m.best_iteration for m in models]

    def predict(self, X):
        return np.mean([m.predict(X) for m in self.models], axis=0).astype(np.float32)

    def importance(self):
        agg = {}
        for m in self.models:
            for k, v in m.importance():
                agg[k] = agg.get(k, 0.0) + v / len(self.models)
        return sorted(agg.items(), key=lambda t: -t[1])

    def save(self, path):
        for i, m in enumerate(self.models):
            m.save(str(path).replace(".json", f"_{i}.json"))


def train_stage2(X, y, Xv, yv, rounds, names):
    models = []
    for i in range(len(STAGE2_VARIANTS)):
        m = train_one(X, y, Xv, yv, rounds, names, device=_device(i), params=STAGE2_VARIANTS[i])
        print(f"    stage2 variant {i}: best_iter {m.best_iteration}{_ram()}", flush=True)
        models.append(m)
        gc.collect()
    return Ensemble(models)

In [ ]:
%%writefile "{SRC_DIR}/ber/decide.py"
"""Turn pair probabilities into per-S1 match sets.

1. Exclusivity: an S2/S3 record belongs to at most one S1 (true for every
   training label), so each S2/S3 record keeps only its best-scoring S1.
2. Expected-F0.5 set selection per S1. For one entity with k predicted and T
   true matches, F0.5 = 1.25*TP / (0.25*T + k). With calibrated probabilities
   p_1 >= p_2 >= ..., predicting the top k gives approximately
       E[F](k) = 1.25 * sum(p_1..p_k) / (0.25 * (sum(p) + miss) + k)
   and predicting nothing scores P(no true match) = prod(1 - p_i) * (1 - q).
   `miss` (true matches outside the candidate list) and `q` are tuned on
   validation, as is a probability floor.
"""
import numpy as np
import polars as pl


def exclusive(pairs, score="p"):
    """Keep, for every S2/S3 record, only its highest-scoring S1."""
    best = pl.col(score) == pl.col(score).max().over("s23_row")
    return pairs.filter(best).unique(subset=["s23_row"], keep="first")


def select_sets(pairs, score="p", miss=0.3, empty_bias=1.0, floor=0.05):
    """Expected-F0.5 subset per S1. `pairs` must already be exclusive.
    Returns the chosen (s1_row, s23_row) pairs."""
    df = pairs.filter(pl.col(score) >= floor).sort(["s1_row", score], descending=[False, True])
    if len(df) == 0:
        return df.select("s1_row", "s23_row")
    s1 = df["s1_row"].to_numpy()
    p = df[score].to_numpy().astype(np.float64)
    starts = np.flatnonzero(np.r_[True, s1[1:] != s1[:-1]])
    grp = np.repeat(np.arange(len(starts)), np.diff(np.r_[starts, len(s1)]))
    k = np.arange(len(s1)) - starts[grp] + 1
    cum = np.cumsum(p)
    cum_tp = cum - np.r_[0, cum][starts][grp]
    tot = np.add.reduceat(p, starts)[grp]
    ef = 1.25 * cum_tp / (0.25 * (tot + miss) + k)
    # probability that the entity truly has no match
    log_none = np.add.reduceat(np.log1p(-np.clip(p, 0, 1 - 1e-9)), starts)
    e_empty = np.exp(log_none) * empty_bias
    best_k = np.zeros(len(starts), dtype=np.int64)
    best_v = e_empty.copy()
    # max over k within each group
    order = np.lexsort((-ef, grp))
    first = order[np.r_[True, grp[order][1:] != grp[order][:-1]]]
    gbest = grp[first]
    better = ef[first] > best_v[gbest]
    best_k[gbest[better]] = k[first][better]
    keep = k <= best_k[grp]
    return df.filter(pl.Series(keep)).select("s1_row", "s23_row")


def threshold_sets(pairs, score="p", t=0.5):
    return pairs.filter(pl.col(score) >= t).select("s1_row", "s23_row")

In [ ]:
%%writefile "{SRC_DIR}/ber/evaluate.py"
"""Entity-level macro F0.5, exactly as the challenge scores it."""
import numpy as np
import polars as pl


def macro_f05(pred_pairs, gt_pairs, s1_ids):
    """pred_pairs / gt_pairs: frames with columns (s1, s23) as entity-id strings.
    s1_ids: every S1 entity in the evaluation set (singletons included).
    Returns dict with macro F0.5, mean precision/recall and singleton stats.
    """
    base = pl.DataFrame({"s1": s1_ids}).unique()
    ids = set(base["s1"].to_list())
    pred = pred_pairs.select("s1", "s23").unique().filter(pl.col("s1").is_in(ids))
    gt = gt_pairs.select("s1", "s23").unique().filter(pl.col("s1").is_in(ids))
    tp = pred.join(gt, on=["s1", "s23"]).group_by("s1").len("tp")
    n_pred = pred.group_by("s1").len("np")
    n_true = gt.group_by("s1").len("nt")
    df = (base.join(n_pred, on="s1", how="left").join(n_true, on="s1", how="left")
          .join(tp, on="s1", how="left").fill_null(0))
    npred, ntrue, ntp = (df[c].to_numpy().astype(np.float64) for c in ("np", "nt", "tp"))
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.where(npred > 0, ntp / npred, 0.0)
        r = np.where(ntrue > 0, ntp / ntrue, 0.0)
        f = np.where((p + r) > 0, 1.25 * p * r / (0.25 * p + r), 0.0)
    f = np.where((npred == 0) & (ntrue == 0), 1.0, f)
    both = (npred > 0) & (ntrue > 0)
    return {
        "macro_f05": float(f.mean()),
        "precision": float(p[both].mean()) if both.any() else 0.0,
        "recall": float(r[both].mean()) if both.any() else 0.0,
        "n": int(len(f)),
        "singleton_acc": float(((npred == 0) & (ntrue == 0)).sum() / max((ntrue == 0).sum(), 1)),
        "pred_empty_rate": float((npred == 0).mean()),
    }

In [ ]:
%%writefile "{SRC_DIR}/ber/pipeline.py"
"""End-to-end pipeline: normalise -> block -> features -> 2-stage GBDT -> decide -> write.

Entry point: run(cfg). Every stage caches to cfg["work_dir"], so a crashed or
restarted Kaggle session resumes where it stopped.
"""
import gc
import hashlib
import json
import time
from pathlib import Path

import numpy as np
import polars as pl

from . import block, decide, geo, model
from .evaluate import macro_f05
from .features import group_features, pair_features, record_stats, write_context
from .prep import load_ground_truth, prepare

DEFAULT_CFG = {
    "data_dir": "dataset",          # contains train/ and test/
    "work_dir": "work",
    "output_dir": "output",
    "workers": None,                # normalisation processes (None = all cores)
    "dev_states": None,             # e.g. ["us_tx", "in_ka"]: run on a state slice for quick checks
    "train_s1_frac": 1.0,           # fraction of train S1 entities used to fit (memory/time knob)
    "holdout_frac": 0.2,            # S1 entities held out for early stopping + decision tuning
    # Test has ~34-39% S2/S3 records whose business is absent from S1, train ~27%.
    # Removing this share of train S1 entities (their S2/S3 records stay as distractors)
    # makes training and the holdout face test-like distractor density.
    "orphan_frac": 0.15,
    "feature_chunk": 2_000_000,
    "stage1_rounds": 2000,
    "stage2_rounds": 2000,
    "n_folds": 3,
    "seed": 42,
    "xgb_params": {},               # overrides for model.XGB_PARAMS (e.g. learning_rate)
}


def log(msg, t0=[time.time()]):
    try:
        import psutil
        vm = psutil.virtual_memory()
        mem = f" | RAM {psutil.Process().memory_info().rss / 1e9:.1f} GB used by us, {vm.available / 1e9:.1f} GB free"
    except Exception:
        mem = ""
    print(f"[{time.time() - t0[0]:7.0f}s] {msg}{mem}", flush=True)


# ---------------------------------------------------------------------------
# data
# ---------------------------------------------------------------------------
def load_split(cfg, split):
    """Normalised S1 and S2+S3 frames for a split (restricted to dev_states if set)."""
    paths = prepare(cfg["data_dir"], Path(cfg["work_dir"]) / "norm", split, workers=cfg["workers"])
    s1 = pl.scan_parquet(paths["source1"])
    s23 = pl.concat([
        pl.scan_parquet(paths["source2"]).with_columns(pl.lit(0, pl.Int8).alias("is_s3")),
        pl.scan_parquet(paths["source3"]).with_columns(pl.lit(1, pl.Int8).alias("is_s3")),
    ])
    if cfg["dev_states"]:
        s1 = s1.filter(pl.col("state").is_in(cfg["dev_states"]))
        s23 = s23.filter(pl.col("state").is_in(cfg["dev_states"]) | (pl.col("state") == ""))
    drop = ["raw_name", "raw_addr", "country_raw"]
    s1, s23 = s1.drop(drop).collect(), s23.drop(drop).collect()
    # fill missing S2/S3 states from address tokens (table learned from this split's S1)
    s23 = geo.infer_states(s23, geo.state_table(s1))
    filled = s23.group_by("country").agg(pl.col("state_inferred").sum()).sort("country").rows()
    log(f"{split}: states inferred for S2/S3 records without one: {filled}")
    return s1, s23


def cached_candidates(cfg, split, s1, s23):
    # cache key covers every setting that changes the candidate set, so a rerun with new
    # blocking parameters never silently reuses stale candidates
    key = json.dumps([block.CFG, block.COUNTRY_CFG, geo.MIN_COUNT, geo.MIN_PURITY, geo.MIN_MARGIN,
                      cfg["dev_states"], cfg["orphan_frac"] if split == "train" else None], sort_keys=True)
    path = Path(cfg["work_dir"]) / f"cands_{split}_{hashlib.md5(key.encode()).hexdigest()[:10]}.parquet"
    if path.exists():
        return pl.read_parquet(path)
    log(f"blocking {split}: S1 {len(s1):,}  S2/S3 {len(s23):,}")
    c = block.generate_candidates(s1, s23)
    c.write_parquet(path)
    return c


# ---------------------------------------------------------------------------
# features
# ---------------------------------------------------------------------------
BLOCK_COLS = ["blk_full", "blk_full_rank", "blk_name", "blk_name_rank", "is_s3"]


def base_features(pairs, s1, s23, chunk):
    """Feature matrix for `pairs`, filled chunk by chunk into one preallocated float32
    array. The last model.N_CONTEXT columns are reserved for stage-2 context, so adding
    it later needs no copy (peak RAM ~ one matrix)."""
    s1, s23, idf = record_stats(s1, s23)
    ctx = group_features(pairs.select("s1_row", "s23_row", "blk_full"), "blk_full").drop(
        "s1_row", "s23_row", "blk_full")
    blk = pairs.select(BLOCK_COLS)
    X, names = None, None
    for i in range(0, len(pairs), chunk):
        f = pl.concat([pair_features(pairs.slice(i, chunk), s1, s23, idf),
                       blk.slice(i, chunk), ctx.slice(i, chunk)], how="horizontal")
        if X is None:
            names = f.columns
            X = np.empty((len(pairs), len(names) + model.N_CONTEXT), dtype=np.float32)
        X[i:i + len(f), :len(names)] = f.to_numpy().astype(np.float32, copy=False)
        log(f"  features {min(i + chunk, len(pairs)):,}/{len(pairs):,}")
    return X, names


def with_context(X, names, pairs, p1, s23):
    """Write stage-2 context into the reserved columns of X (in place)."""
    ctx_names = write_context(X, len(names), pairs, p1, s23)
    assert len(ctx_names) == model.N_CONTEXT == X.shape[1] - len(names), (len(ctx_names), X.shape)
    gc.collect()
    log("stage-2 context features written")
    return X, names + ctx_names


# ---------------------------------------------------------------------------
# [sub3a3] training fraction: 0.5 when its EXTRA memory over the Kaggle-proven 0.3 fits
# ---------------------------------------------------------------------------
# Training holds X (float32, pairs x columns) plus copies of the fit and holdout-A rows (X[tr], X[va] per fold in
# stage 1; X2[fit], X2[holdout A] in stage 2). Estimated peak = (all pairs + fit/holdout-A pairs) x columns x 4 B.
# 0.3 is known to fit on Kaggle, so 0.5 is used only if its extra estimated peak uses at most half of the RAM left
# after 0.3's estimated peak and a reserve (polars frames, XGBoost buffers). Otherwise 0.3, exactly as before.
FEATURE_COLS = 88      # stage-1 + stage-2 columns of X on the friend's feature set
RESERVE_GB = 3.0


def _roles(cfg, u, frac):
    h = cfg["holdout_frac"]
    return np.where(u < h / 2, 1,                       # holdout A
           np.where(u < h, 2,                           # holdout B
           np.where(u < h + frac * (1 - h), 0, 3)))


def _estimate(role, s1_row, s23_row):
    r = role[s1_row]
    used = np.unique(s23_row[r < 3])
    kept = (r < 3) | np.isin(s23_row, used)             # same rule as the pairs filter in train()
    n = int(kept.sum())
    n_copied = int(((r == 0) | (r == 1))[kept].sum())
    return n, (n + n_copied) * FEATURE_COLS * 4


def choose_train_frac(cfg, u, s1_row, s23_row):
    """-> (frac, role). Tries the options above the floor (largest first); the floor (smallest option, proven on
    Kaggle) is used when none passes. Every estimate is logged so the next version can be tuned from real numbers."""
    options = sorted(set(cfg.get("train_s1_frac_options") or [cfg["train_s1_frac"]]), reverse=True)
    floor = options[-1]
    try:
        import psutil
        free = psutil.virtual_memory().available
    except Exception:
        free = None
    role0 = _roles(cfg, u, floor)
    n0, peak0 = _estimate(role0, s1_row, s23_row)
    headroom = (free - peak0 - RESERVE_GB * 1e9) if free is not None else -1
    log(f"train_s1_frac {floor} (proven): {n0:,} pairs, estimated peak {peak0 / 1e9:.1f} GB; free RAM "
        f"{(free or 0) / 1e9:.1f} GB -> headroom {headroom / 1e9:.1f} GB after a {RESERVE_GB:.0f} GB reserve")
    for frac in options[:-1]:
        role = _roles(cfg, u, frac)
        n, peak = _estimate(role, s1_row, s23_row)
        extra = peak - peak0
        fits = headroom > 0 and extra <= 0.5 * headroom
        log(f"train_s1_frac {frac}: {n:,} pairs (+{(n - n0) / max(n0, 1):.1%}), estimated extra peak "
            f"{extra / 1e9:.1f} GB vs half the headroom {max(headroom, 0) / 2e9:.1f} GB -> {'fits' if fits else 'too big'}")
        if fits:
            log(f"using train_s1_frac {frac}")
            return frac, role
    log(f"using train_s1_frac {floor} (the proven setting)")
    return floor, role0


# ---------------------------------------------------------------------------
# training
# ---------------------------------------------------------------------------
def train(cfg):
    work = Path(cfg["work_dir"])
    model.XGB_PARAMS.update(cfg.get("xgb_params") or {})
    log(f"GPUs: {model.n_gpus()}  xgb params: {model.XGB_PARAMS}")
    s1, s23 = load_split(cfg, "train")
    gt, _ = load_ground_truth(Path(cfg["data_dir"]) / "train" / "train_ground_truth.tsv")
    if cfg["orphan_frac"] > 0:
        keep = np.random.default_rng(cfg["seed"] + 1).random(len(s1)) >= cfg["orphan_frac"]
        s1 = s1.filter(pl.Series(keep))
        matched = gt.filter(pl.col("s1").is_in(s1["entity_id"].implode()) &
                            pl.col("s23").is_in(s23["entity_id"].implode()))["s23"].n_unique()
        log(f"orphaning: removed {int((~keep).sum()):,} train S1 entities; S2/S3 without a match in S1 "
            f"now {1 - matched / len(s23):.1%} (test ~34-39%)")
    cands = cached_candidates(cfg, "train", s1, s23)
    s1_ids, s23_ids = s1["entity_id"], s23["entity_id"]

    # labels
    cands = cands.with_columns(
        s1_ids.gather(cands["s1_row"]).alias("s1"), s23_ids.gather(cands["s23_row"]).alias("s23"),
        s23["is_s3"].gather(cands["s23_row"]).alias("is_s3"),
    ).join(gt.with_columns(pl.lit(1, pl.Int8).alias("y")), on=["s1", "s23"], how="left") \
     .with_columns(pl.col("y").fill_null(0))
    gt_local = gt.filter(pl.col("s1").is_in(s1_ids.implode()))
    rec = cands["y"].sum() / max(len(gt_local), 1)
    log(f"candidates {len(cands):,} ({len(cands) / len(s23):.2f}/S2S3, {len(cands) / len(s1):.1f}/S1), "
        f"pair recall {rec:.4f}")

    # entity split: holdout (A = early stopping, B = decision tuning/report), fit, unused
    rng = np.random.default_rng(cfg["seed"])
    u = rng.random(len(s1))
    frac, role = choose_train_frac(cfg, u, cands["s1_row"].to_numpy(), cands["s23_row"].to_numpy())
    cfg["train_s1_frac"] = frac   # the report records the fraction actually used
    cands = cands.with_columns(pl.Series("role", role[cands["s1_row"].to_numpy()]).cast(pl.Int8))
    # keep pairs of used entities plus the competitor pairs of their S2/S3 records
    used_s23 = cands.filter(pl.col("role") < 3)["s23_row"].unique()
    pairs = cands.filter((pl.col("role") < 3) | pl.col("s23_row").is_in(used_s23.implode()))
    # string ids were only needed for the label join; rows index s1_ids/s23_ids from here on
    pairs = pairs.drop("s1", "s23")
    del cands
    gc.collect()

    X, names = base_features(pairs, s1, s23, cfg["feature_chunk"])
    # after features only these columns are still used (context, segments, outputs)
    s1 = s1.select("entity_id", "country")
    s23 = s23.select("entity_id", "core", "addr", "house", "is_s3")
    gc.collect()
    y = pairs["y"].to_numpy()
    r = pairs["role"].to_numpy()
    s1_row = pairs["s1_row"].to_numpy()
    log(f"feature matrix {X.shape}, positives {int(y.sum()):,}")

    # stage 1: out-of-fold on fit rows
    fit = r == 0
    p1, m1 = model.stage1_oof(X[:, :len(names)], y, s1_row, fit, names, rounds=cfg["stage1_rounds"],
                              n_folds=cfg["n_folds"])
    X2, names2 = with_context(X, names, pairs, p1, s23)
    gc.collect()
    # stage 2: fit rows, early stopping on holdout A (two variants, averaged)
    m2 = model.train_stage2(X2[fit], y[fit], X2[r == 1], y[r == 1], cfg["stage2_rounds"], names2)
    p2 = m2.predict(X2)
    del X, X2   # ~10 GB; nothing below needs the feature matrix
    gc.collect()
    log("stage 2 done, feature matrix released")
    pairs = pairs.with_columns(pl.Series("p1", p1), pl.Series("p", p2))
    pairs.select("s1_row", "s23_row", "y", "role", "p1", "p").write_parquet(work / "train_scored.parquet")
    imp = m2.importance()[:25]
    log("stage2 top features: " + ", ".join(f"{n}={g:.0f}" for n, g in imp))

    # decision tuning on holdout B (full candidate context, scored on B entities only)
    b_ids = s1_ids.filter(pl.Series(role == 2))
    params, report = tune_decision(pairs, b_ids, gt_local, s1_ids, s23_ids, s1, s23)
    report.update({"pair_recall": float(rec), "n_candidates": int(len(pairs)),
                   "orphan_frac": cfg["orphan_frac"], "train_s1_frac": cfg["train_s1_frac"],
                   "stage1_iters": [int(m.best_iteration) for m in m1], "stage2_iter": m2.best_iteration,
                   "features": names2, "decision": params})
    for i, m in enumerate(m1):
        m.save(work / f"stage1_{i}.json")
    m2.save(work / "stage2.json")
    (work / "train_report.json").write_text(json.dumps(report, indent=2))
    return m1, m2, params, report


def _to_ids(sel, s1_ids, s23_ids):
    return pl.DataFrame({"s1": s1_ids.gather(sel["s1_row"]), "s23": s23_ids.gather(sel["s23_row"])})


def segment_report(sel, eval_ids, gt, s1_ids, s23_ids, s1, s23):
    """Macro F0.5 by S1 country and by whether the entity has an address-less true match,
    plus pair recall split by S2/S3 address presence. The weak segments measured on
    submission 1 were address-less records and large Indian states."""
    pred = _to_ids(sel, s1_ids, s23_ids)
    ctry = pl.DataFrame({"s1": s1_ids, "country": s1["country"]})
    noaddr = s23.filter(pl.col("addr") == "")["entity_id"]
    ent = pl.DataFrame({"s1": eval_ids}).join(ctry, on="s1", how="left")
    has_na = gt.filter(pl.col("s23").is_in(noaddr.implode()))["s1"].unique()
    out = {}
    for c in sorted(ent["country"].unique().to_list()):
        ids = ent.filter(pl.col("country") == c)["s1"]
        out[f"country={c}"] = macro_f05(pred, gt, ids)["macro_f05"]
    for flag, name in [(True, "entity has address-less match"), (False, "entity has no address-less match")]:
        ids = ent.filter(pl.col("s1").is_in(has_na.implode()) == flag)["s1"]
        if len(ids):
            out[name] = macro_f05(pred, gt, ids)["macro_f05"]
    g = gt.filter(pl.col("s1").is_in(eval_ids.implode())).with_columns(pl.col("s23").is_in(noaddr.implode()).alias("na"))
    hit = g.join(pred, on=["s1", "s23"], how="semi")
    for flag, name in [(True, "pair recall, address-less S2/S3"), (False, "pair recall, S2/S3 with address")]:
        n = g.filter(pl.col("na") == flag).height
        out[name] = hit.filter(pl.col("na") == flag).height / max(n, 1)
    for k, v in out.items():
        log(f"    {k:36s} {v:.5f}")
    return out


def tune_decision(pairs, eval_ids, gt, s1_ids, s23_ids, s1=None, s23=None):
    """Grid-search the decision rule on the evaluation entities; returns (params, report)."""
    ev = pairs.filter(pl.col("s1_row").is_in(
        pl.Series(np.flatnonzero(s1_ids.is_in(eval_ids.implode()).to_numpy())).implode()))
    # exclusivity is decided with every competitor present, then restricted to eval entities
    excl = decide.exclusive(pairs.filter(pl.col("s23_row").is_in(ev["s23_row"].unique().implode())), "p")
    excl = excl.filter(pl.col("s1_row").is_in(ev["s1_row"].unique().implode()))
    score = lambda sel: macro_f05(_to_ids(sel, s1_ids, s23_ids), gt, eval_ids)
    rows = []
    for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
        rows.append(({"rule": "threshold", "t": t, "exclusive": False}, score(decide.threshold_sets(ev, "p", t))))
        rows.append(({"rule": "threshold", "t": t, "exclusive": True}, score(decide.threshold_sets(excl, "p", t))))
    for miss in [0.0, 0.2, 0.5]:
        for eb in [0.8, 1.0, 1.25]:
            for floor in [0.02, 0.1]:
                prm = {"rule": "expected_f", "miss": miss, "empty_bias": eb, "floor": floor, "exclusive": True}
                rows.append((prm, score(decide.select_sets(excl, "p", miss, eb, floor))))
    rows.sort(key=lambda t: -t[1]["macro_f05"])
    for prm, res in rows[:6]:
        log(f"  {prm} -> F0.5 {res['macro_f05']:.5f}  P {res['precision']:.4f}  R {res['recall']:.4f}  "
            f"empty-acc {res['singleton_acc']:.3f}")
    best, res = rows[0]
    oracle = score(ev.filter(pl.col("y") == 1))
    log(f"best decision {best}: holdout-B macro F0.5 = {res['macro_f05']:.5f} "
        f"(blocking ceiling {oracle['macro_f05']:.5f})")
    grid = [{**prm, **{k: round(v, 5) for k, v in r.items() if k in ("macro_f05", "precision", "recall", "singleton_acc")}}
            for prm, r in rows[:12]]
    seg = {}
    if s1 is not None:
        chosen = apply_decision(excl if best.get("exclusive") else ev, {**best, "exclusive": False})
        log("holdout-B by segment (chosen decision):")
        seg_chosen = segment_report(chosen, eval_ids, gt, s1_ids, s23_ids, s1, s23)
        log("holdout-B blocking ceiling by segment:")
        seg_ceiling = segment_report(ev.filter(pl.col("y") == 1), eval_ids, gt, s1_ids, s23_ids, s1, s23)
        seg = {"chosen": seg_chosen, "ceiling": seg_ceiling}
    return best, {"holdout_b": res, "holdout_b_ceiling": oracle, "decision_grid": grid, "segments": seg}


# ---------------------------------------------------------------------------
# inference + outputs
# ---------------------------------------------------------------------------
def apply_decision(pairs, params):
    sel = decide.exclusive(pairs, "p") if params.get("exclusive") else pairs
    if params["rule"] == "threshold":
        return decide.threshold_sets(sel, "p", params["t"])
    return decide.select_sets(sel, "p", params["miss"], params["empty_bias"], params["floor"])


def predict_test(cfg, m1, m2, params):
    s1, s23 = load_split(cfg, "test")
    cands = cached_candidates(cfg, "test", s1, s23)
    cands = cands.with_columns(s23["is_s3"].gather(cands["s23_row"]).alias("is_s3"))
    log(f"test candidates {len(cands):,} ({len(cands) / len(s1):.1f}/S1)")
    X, names = base_features(cands, s1, s23, cfg["feature_chunk"])
    s1 = s1.select("entity_id", "country")
    s23 = s23.select("entity_id", "core", "addr", "house", "is_s3")
    gc.collect()
    p1 = model.stage1_predict(m1, X[:, :len(names)])
    log("test stage 1 scored")
    X2, _ = with_context(X, names, cands, p1, s23)
    gc.collect()
    p = m2.predict(X2)
    del X, X2
    gc.collect()
    log("test stage 2 scored, feature matrix released")
    cands = cands.with_columns(pl.Series("p", p))
    chosen = apply_decision(cands, params)
    write_outputs(cfg, s1["entity_id"], s23["entity_id"], cands, chosen)
    # per-country prediction profile: France has no labels, so this is its only health check
    prof = (pl.DataFrame({"country": s1["country"]}).with_row_index("s1_row")
            .join(chosen.group_by("s1_row").len("n").with_columns(pl.col("s1_row").cast(pl.UInt32)),
                  on="s1_row", how="left").with_columns(pl.col("n").fill_null(0))
            .group_by("country").agg((pl.col("n") == 0).mean().alias("empty_rate"), pl.col("n").mean().alias("mean_matches"))
            .sort("country"))
    log(f"test prediction profile by country (empty rate, mean matches): {prof.rows()}")
    (Path(cfg["work_dir"]) / "test_report.json").write_text(json.dumps({
        "n_s1": len(s1), "n_s23": len(s23), "n_candidates": len(cands),
        "candidates_per_s1": len(cands) / max(len(s1), 1), "n_matches": len(chosen),
        "profile_by_country": {c: {"empty_rate": e, "mean_matches": m} for c, e, m in prof.rows()}}, indent=2))
    return cands, chosen


def _id_lists(pairs, s1_ids, s23_ids, col):
    """One row per S1 (all of them), comma-joined S2/S3 ids (empty when none)."""
    lists = (pl.DataFrame({"source1_entity_id": s1_ids.gather(pairs["s1_row"]),
                           "id": s23_ids.gather(pairs["s23_row"])})
             .unique().sort("id")
             .group_by("source1_entity_id").agg(pl.col("id").str.join(",").alias(col)))
    return (pl.DataFrame({"source1_entity_id": s1_ids}).join(lists, on="source1_entity_id", how="left")
            .with_columns(pl.col(col).fill_null("")))


def write_outputs(cfg, s1_ids, s23_ids, cands, chosen):
    out = Path(cfg["output_dir"])
    out.mkdir(parents=True, exist_ok=True)
    # candidate_pairs.tsv = exactly the pairs the final model scored
    _id_lists(cands, s1_ids, s23_ids, "candidate_entity_ids").write_csv(
        out / "candidate_pairs.tsv", separator="\t", quote_style="never")
    m = _id_lists(chosen, s1_ids, s23_ids, "matched_entity_ids")
    m.write_csv(out / "matching_results.tsv", separator="\t", quote_style="never")
    n_empty = int((m["matched_entity_ids"] == "").sum())
    log(f"wrote {out}/matching_results.tsv: {len(m):,} S1 rows, {n_empty:,} empty "
        f"({n_empty / len(m):.1%}), {len(chosen):,} matches")


def run(cfg=None):
    cfg = {**DEFAULT_CFG, **(cfg or {})}
    Path(cfg["work_dir"]).mkdir(parents=True, exist_ok=True)
    m1, m2, params, report = train(cfg)
    gc.collect()
    predict_test(cfg, m1, m2, params)
    return report

In [ ]:
%%writefile "{SRC_DIR}/run.py"
"""Reproduce output/matching_results.tsv and output/candidate_pairs.tsv end to end.

    python src/run.py --data-dir ../../dataset --work-dir work --output-dir output

Add --dev-states us_tx,in_ka for a quick sliced run.
"""
import argparse
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from ber import pipeline  # noqa: E402

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", default="dataset")
    ap.add_argument("--work-dir", default="work")
    ap.add_argument("--output-dir", default="output")
    ap.add_argument("--train-s1-frac", type=float, default=pipeline.DEFAULT_CFG["train_s1_frac"])
    ap.add_argument("--dev-states", default=None)
    ap.add_argument("--workers", type=int, default=None)
    a = ap.parse_args()
    report = pipeline.run({
        "data_dir": a.data_dir, "work_dir": a.work_dir, "output_dir": a.output_dir,
        "train_s1_frac": a.train_s1_frac, "workers": a.workers,
        "dev_states": a.dev_states.split(",") if a.dev_states else None,
    })
    print(json.dumps(report["holdout_b"], indent=2))

In [ ]:
sys.path.insert(0, str(SRC_DIR))
for m in [m for m in list(sys.modules) if m == "ber" or m.startswith("ber.")]:
    del sys.modules[m]
from ber import pipeline, normalize, model
n_gpu = model.n_gpus()
print("GPUs visible to XGBoost:", n_gpu)
if IS_KAGGLE and n_gpu == 0:
    print("!!! WARNING: running on Kaggle but no GPU detected. Settings (top right) > "
          "Accelerator > GPU T4 x2, then Save and re-run. Falling back to CPU for now.")
elif IS_KAGGLE and n_gpu < 2:
    print(f"NOTE: only {n_gpu} GPU visible (expected 2). Folds will still train, just less in parallel.")
for n in ["गैलेक्सी कंसल्टेंट्स प्राइवेट लिमिटेड", "Galaxy Consultants Private Limited",
          "ಇನೋವೇಟಿವ್ ಎಕ್ಸ್‌ಪೋರ್ಟ್ಸ್ ಲಿಮಿಟೆಡ್", "Innovative Exports Limited", "31eme Amicale SARL"]:
    r = normalize.normalize_name(n)
    print(f"{n:40s} -> {r['core']:26s} {' '.join(normalize.phonetic_key(t) for t in r['core'].split())}")
print(normalize.normalize_address("N°139 R. BAUDUCHEU, BORDEAUX, Gironde", "France"))
print(normalize.normalize_address("13-14 ROOP NAGAR, INDORE, मध्य प्रदेश", "India"))

## 2. Train: normalise → regions → block → features → stage 1 (OOF, 2 GPUs) → stage 2 → tune decision

In [ ]:
cfg = {**pipeline.DEFAULT_CFG, **CFG}
Path(cfg["work_dir"]).mkdir(parents=True, exist_ok=True)
m1, m2, params, report = pipeline.train(cfg)
print(json.dumps({k: report[k] for k in ["pair_recall", "holdout_b", "holdout_b_ceiling", "decision", "segments"]}, indent=2))

## 3. Test: normalise → regions → block → features → score → decide → write TSVs

In [ ]:
import gc; gc.collect()
test_pairs, chosen = pipeline.predict_test(cfg, m1, m2, params)
del test_pairs; gc.collect()

## 4. Official validator

In [ ]:
out = Path(cfg["output_dir"])
val = RESOURCE_DIR / "utils" / "validate_submission.py"
res = subprocess.run([sys.executable, str(val), "--matching", str(out / "matching_results.tsv"),
                      "--candidate", str(out / "candidate_pairs.tsv"), "--test-dir", str(DATA_DIR / "test")],
                     capture_output=True, text=True, cwd=str(RESOURCE_DIR))
print(res.stdout[-4000:], res.stderr[-2000:])
if cfg["dev_states"]:
    print()
    print("This is a DEV run (states:", cfg["dev_states"], ") so a FAIL above listing 'missing required "
          "S1 rows' is EXPECTED - only a slice of test S1 entities were predicted on purpose. "
          "Set DEV = False for the real, validator-passing run.")
else:
    assert res.returncode == 0, "validator failed on the FULL run - do not submit, read the errors above"
    print("PASS on the full run.")

## 5. Package: zip only (leaderboard file is `output/matching_results.tsv` inside it)

In [ ]:
work = Path(cfg["work_dir"])
rep = json.loads((work / "train_report.json").read_text())
trep = json.loads((work / "test_report.json").read_text())
hb = rep["holdout_b"]
grid = "| rule | params | macro F0.5 | P | R |\n| --- | --- | --- | --- | --- |\n" + "\n".join(
    f"| {g['rule']} | { {k: v for k, v in g.items() if k not in ('rule', 'macro_f05', 'precision', 'recall', 'singleton_acc')} } "
    f"| {g['macro_f05']:.5f} | {g['precision']:.4f} | {g['recall']:.4f} |" for g in rep["decision_grid"])
seg = rep.get("segments") or {}
seg_md = "| segment | achieved | blocking ceiling |\n| --- | --- | --- |\n" + "\n".join(
    f"| {k} | {v:.4f} | {seg['ceiling'].get(k, float('nan')):.4f} |" for k, v in seg.get("chosen", {}).items())
prof = trep.get("profile_by_country", {})
seg_md += "\n\nTest predictions by country (no labels; health check): " + ", ".join(
    f"{c}: {v['empty_rate']:.1%} empty, {v['mean_matches']:.2f} matches per S1" for c, v in prof.items())
fill = {
    "<<TEAM>>": TEAM, "<<MEMBERS>>": MEMBERS, "<<DATE>>": datetime.date.today().isoformat(),
    "<<F05>>": f"{hb['macro_f05']:.4f}", "<<PREC>>": f"{hb['precision']:.4f}", "<<REC>>": f"{hb['recall']:.4f}",
    "<<SING>>": f"{hb['singleton_acc']:.1%}", "<<CEIL>>": f"{rep['holdout_b_ceiling']['macro_f05']:.4f}",
    "<<RECALL>>": f"{rep['pair_recall']:.2%}", "<<NPAIRS_TEST>>": f"{trep['n_candidates']:,}",
    "<<PER_S1_TEST>>": f"{trep['candidates_per_s1']:.1f}", "<<SEGMENTS>>": seg_md,
    "<<DECISION_TABLE>>": "Decision rules on held-out S1 entities (best first):\n\n" + grid,
}
doc = "# ML Challenge 2026: Business Entity Resolution Solution\n\n**Team Name:** <<TEAM>>\n**Team Members:** <<MEMBERS>>\n**Submission Date:** <<DATE>>\n\n---\n\n## 1. Executive Summary\nWe resolve S2/S3 records to S1 businesses in four stages:\n1. Script-aware normalisation, including a phonetic key that matches Hindi and Kannada spellings of English words.\n2. Blocking in the S2/S3 \u2192 S1 direction with IDF-cosine token retrieval inside each (country, state).\n3. A two-stage XGBoost scorer whose second stage uses competitor context and \"sibling\" evidence from the S1's other confident records.\n4. A per-entity decision rule that maximises *expected* F0.5 and enforces the fact that every S2/S3 record belongs to at most one S1.\n\nTraining and validation are made test-like by removing 15% of train S1 entities, so their S2/S3 records become distractors: test has ~34\u201339% S2/S3 records whose business is absent from S1, train only ~27%. On held-out training entities under that setting we reach macro F0.5 **<<F05>>**, against a blocking ceiling of <<CEIL>>. The first submission scored 0.9687 on the public leaderboard.\n\n---\n\n## 2. Methodology\n\n### 2.1 Problem Analysis\nMeasured on the training files:\n- **Exclusivity.** Every S2/S3 record matches at most one S1 (7.64M of 10.3M S2/S3 records are matched, none twice).\n- **Match counts.** 5.6% of S1 are singletons; the median is 3 matches and the maximum 11.\n- **Addresses.** They contain no ZIP/PIN codes; the digits are house or plot numbers, which the noise generator perturbs arithmetically (15022 \u2192 15023).\n- **Indic scripts.** 10\u201315% of S2/S3 rows contain Indic script across nine scripts. Hindi and Kannada names are phonetic spellings of English words (\"\u0917\u0948\u0932\u0947\u0915\u094d\u0938\u0940 \u0915\u0902\u0938\u0932\u094d\u091f\u0947\u0902\u091f\u094d\u0938\" = \"Galaxy Consultants\"), and state names appear in native script. Zero-width joiners sit inside Kannada words.\n- **Name noise.** Legal forms move or swap (Pvt/Private, Ltd/Limited, LLC placed first), DBA prefixes appear, and some names are domains or handles (\"nexosidea.com\", \"@nizarmarine\"). Typos include leet substitutions (N0rth, 8akery) and filler words are appended (Center, Partners, Service, \"(France)\"). Some names are replaced by random tokens, and some addresses are literally \"None\".\n- **Address noise.** Components are reordered, state names are swapped between full and abbreviated forms, PMB/PO boxes are added, and Telangana \u2194 Andhra Pradesh are mixed. True pairs agree on state 98.4% of the time.\n- **France.** It appears only in test (15% of test S1). Sources swap region and d\u00e9partement (Gironde \u2194 Nouvelle-Aquitaine) and abbreviate street types (R., BD, IMP., Rte). 35% of French S2/S3 records carry no region at all, only a city.\n- **Distractors.** Test has 5.75 S2/S3 records per S1 against 4.68 in train, and its blocking-score distribution is shifted down: unmixing it puts test's share of S2/S3 records without an S1 match at ~34\u201339%, against 27% in train. Unmatched records are lookalikes (median best cosine 0.62), not noise.\n- **Address-less records are the hardest segment.** They are 4.4% of true pairs but, in submission 1, 54% of all lost pairs and 43% of all false merges: 52% of S1 names are shared with another S1, and orphan S2/S3 records reuse the same names.\n\n### 2.2 Solution Strategy\n**Approach Type:** Blocking + two-stage gradient-boosted classifier + expected-F0.5 set selection.\n\n**Core Innovations:**\n1. **Unified Indic romanisation and phonetic skeleton.** The nine Indic scripts share one Unicode layout, so a single table romanises all of them. A consonant skeleton then makes cross-script names comparable: `\u0915\u0902\u0938\u0932\u094d\u091f\u0947\u0902\u091f\u094d\u0938` \u2192 `knsltnts` \u2190 \"consultants\".\n2. **Reverse-direction blocking.** Blocking goes from S2/S3 to S1, exploiting exclusivity: each S2/S3 record only needs its best few S1 records.\n3. **Competitor-context and sibling stage 2.** It uses out-of-fold stage-1 scores to see how a pair ranks against the other S1 candidates of the same S2/S3 record, and how much the record resembles the S1's most confident other records (S2\u2194S3 triangulation).\n5. **Test-like validation.** Orphaning 15% of train S1 entities reproduces test's distractor density, so the decision rule is tuned against the harder mix it will face.\n4. **Decision rule matched to the metric.** For one entity, F0.5 = 1.25\u00b7TP / (0.25\u00b7T + k), so the rule chooses the prediction set that maximises its expectation, the empty set included.\n\n---\n\n## 3. Candidate Generation (Blocking)\n- **Keys and representation.** Each record becomes a binary IDF-weighted token vector over:\n  - core-name tokens;\n  - phonetic name keys;\n  - the whole name with spaces removed, which also meets domains and handles;\n  - address tokens;\n  - a house-number + street-word key.\n\n  Tokens with document frequency above 20,000 inside a partition are ignored.\n- **Partitions.** Blocking runs within (country, state). Telangana and Andhra Pradesh search each other, and records with no state search their whole country. Country is an open set, so France is blocked like any other country.\n- **Region inference.** S1 always carries a state/region, so a token\u2192state table is learned from S1 (a token must appear \u226520 times and point to one state \u226597% of the time). S2/S3 records missing a state get one when their address tokens agree. Hiding known states and re-inferring them gives 99.998% accuracy on France (98.1% India, 98.6% US); it fills 91% of French records that lacked a region.\n- **Channels** (parameters chosen by a recall-vs-volume sweep on train, not by hand):\n  - *Full channel:* cosine over all tokens; top 8 S1 per S2/S3 record with score \u2265 0.5 \u00d7 best (India +0.5 points recall over top-6/0.6), top 6 / 0.6 for the US where recall is already 99.7%.\n  - *Name-only channel, empty address:* top 25 with score \u2265 0.3 \u00d7 best (address-less recall 75.7% \u2192 84.9%).\n  - *Name-only channel, 1\u20132 address tokens:* top 4 with score \u2265 0.7 \u00d7 best (widening measured no gain).\n- **Implementation.** `sparse_dot_topn` (Apache-2.0) keeps only the top n per row inside the sparse product.\n- **Candidate pairs generated:** <<NPAIRS_TEST>> on test (<<PER_S1_TEST>> per S1).\n- **How true matches were not lost:** the name channel covers missing addresses, the whole-name token covers domains, the state-neighbour rule covers TG/AP, and records with no state fall back to their country. Pair recall on train is **<<RECALL>>**.\n\n---\n\n## 4. Matching Model\n\n**Features (\u224875):**\n- **Name:** ratio, token-set, token-sort, partial ratio, Jaro-Winkler and Levenshtein on full and core names; the same on phonetic keys; token Jaccard and containment; IDF mass of shared phonetic tokens and the rarest shared token; domain-vs-name partial match; legal-form agreement; name lengths; Indic-script flag.\n- **Name rarity and orphan pressure:** how many S1 records and how many S2/S3 records share the exact core name and the phonetic key, and their ratio. About 3.5 S2/S3 records per S1 are expected, so a much higher ratio means other businesses absent from S1 share the name.\n- **Address:** fuzzy ratios, token Jaccard and containment; numeric-token Jaccard; house-number equality, edit distance, absolute and relative difference; state agreement; PMB flag. Missing values are NaN, meaning no evidence.\n- **Blocking and context:** channel scores and ranks, plus each pair's rank, gap and margin among the candidates of its S2/S3 record and of its S1.\n- **Stage 2** adds the same context statistics computed on out-of-fold stage-1 probabilities, the expected number of matches per S1 and per source, the rank within the same source, and **sibling features**: similarity of the record to the S1's two most confident other records (name, address, house number) and their stage-1 scores.\n\nCountry is deliberately not a feature.\n\n**Model type:** XGBoost (`hist`, GPU), two stages. Stage 1 is 3-fold out-of-fold, grouped by S1 entity, with folds trained in parallel on two T4 GPUs. Stage 2 averages two differently shaped boosters (depth 10 and depth 7 with stronger column sampling), trained concurrently on the two GPUs, with early stopping on held-out entities.\n\n**Threshold selection method:**\n1. Each S2/S3 record keeps only its best S1.\n2. Per S1, choose the top-k that maximises 1.25\u00b7\u03a3p_top-k / (0.25\u00b7(\u03a3p + miss) + k), compared against P(no match) \u00d7 bias.\n3. `miss`, `bias` and a probability floor are grid-searched on a disjoint set of held-out S1 entities, alongside plain thresholds as baselines.\n\n---\n\n## 5. Results & Error Analysis\n\n- **F_0.5 Score (macro, held-out train entities):** <<F05>> (precision <<PREC>>, recall <<REC>>, singletons correct <<SING>>)\n- **Blocking ceiling on the same entities:** <<CEIL>>\n- **By segment (held-out entities):**\n\n<<SEGMENTS>>\n- **Common false positives (wrong merges):** near-duplicate S1 branches with the same name whose house numbers differ by a few units; Indic-script names whose transliteration shares generic words with a different business at the same address.\n- **Common false negatives (missed matches):** S2/S3 records with no address and a common or heavily typo'd name; names replaced by random tokens with truncated addresses; \"None | None\" records.\n\n---\n\n## 6. Conclusion\nMeasuring the data carefully paid off more than model complexity: exclusivity, the phonetic nature of the Indic names, and arithmetic house-number noise each became a dedicated component. The most useful lessons were matching the decision rule to the metric and blocking in the S2/S3 \u2192 S1 direction.\n\n---\n\n## Appendix\n\n### A. Code Artefacts\n`code/business_entity_resolution/src/ber/`:\n\n| module | role |\n| --- | --- |\n| `normalize.py` | normalisation |\n| `geo.py` | state/region inference from S1 |\n| `prep.py` | TSV \u2192 parquet |\n| `block.py` | candidate generation |\n| `features.py` | pair and context features |\n| `model.py` | XGBoost stages |\n| `decide.py` | decision rule |\n| `evaluate.py` | exact metric |\n| `pipeline.py` | orchestration |\n\nEntry point: `python src/run.py --data-dir <dataset> --output-dir output`, or run the notebook `erk_sub2.ipynb` end to end. Both write `output/matching_results.tsv` and `output/candidate_pairs.tsv`.\n\n### B. Additional Results\n<<DECISION_TABLE>>\n"
for k, v in fill.items():
    doc = doc.replace(k, v)
(CODE_DIR / "requirements.txt").write_text("# Python 3.10+ ; versions the pipeline was developed and validated with\nnumpy==2.3.1\nscipy==1.18.0\npolars==1.44.2\nrapidfuzz==3.14.6\nsparse_dot_topn==1.2.0   # Apache-2.0, top-n sparse matrix product for blocking\nxgboost==3.4.1           # Apache-2.0, the only learned model (GPU hist)\n")
(CODE_DIR / "README.md").write_text("# Business Entity Resolution \u2014 reproduction\n\n## Setup\n```bash\npip install -r requirements.txt\n```\n\n## Run end to end (data \u2192 blocking \u2192 matching \u2192 output)\n```bash\npython src/run.py --data-dir <student_resource>/dataset --work-dir work --output-dir output\n```\n`--data-dir` is the folder that contains `train/` and `test/`. This writes\n`output/matching_results.tsv` and `output/candidate_pairs.tsv`. Intermediate caches\n(normalised tables, candidates, models) go to `work/`, so a rerun resumes from them.\n\nThe same code runs as the Kaggle notebook `erk_sub2.ipynb`. That notebook writes\nthis `src/` folder itself, so the packaged code is exactly the code that produced the output.\n\nUseful flags:\n- `--dev-states us_tx,in_ka,fr_naq` runs on a slice of states. It takes a few minutes and is\n  meant for smoke tests; its output is not a valid submission.\n- `--train-s1-frac 0.6` sets the share of non-holdout training entities used to fit the\n  models. This is the RAM/time knob; the notebook uses 0.5 when its extra memory over 0.3 fits.\n\nTraining removes 15% of train S1 entities before blocking (`orphan_frac` in\n`pipeline.DEFAULT_CFG`), so their S2/S3 records act as distractors. That matches the test\nset, where ~34-39% of S2/S3 records have no S1 match versus ~27% in train.\n\n## Layout\n| file | role |\n| --- | --- |\n| `src/ber/normalize.py` | name/address normalisation, Indic romanisation, phonetic keys, states |\n| `src/ber/prep.py` | TSV \u2192 normalised parquet (multiprocessing) |\n| `src/ber/geo.py` | fills missing S2/S3 states/regions from a token\u2192state table learned from S1 |\n| `src/ber/block.py` | candidate generation (S2/S3 \u2192 S1 IDF-cosine retrieval per country/state) |\n| `src/ber/features.py` | pairwise similarity and competitor-context features |\n| `src/ber/model.py` | two-stage XGBoost, GPU when available (out-of-fold stage 1, folds in parallel across GPUs) |\n| `src/ber/decide.py` | exclusivity + expected-F0.5 set selection |\n| `src/ber/evaluate.py` | exact entity-level macro F0.5 |\n| `src/ber/pipeline.py` | orchestration, validation tuning, output writing |\n| `src/run.py` | command-line entry point |\n\nSeeds are fixed. GPU histogram training can differ in the last digits between GPU models.\n")
shutil.copy(work / "train_report.json", CODE_DIR / "train_report.json")
shutil.copy(work / "test_report.json", CODE_DIR / "test_report.json")

zpath = BASE / (f"{TEAM}_DEV_submission.zip" if cfg["dev_states"] else f"{TEAM}_submission.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["matching_results.tsv", "candidate_pairs.tsv"]:
        z.write(out / f, f"output/{f}")
    for p in sorted(CODE_DIR.rglob("*")):
        if p.is_file() and "__pycache__" not in p.parts:
            z.write(p, f"code/business_entity_resolution/{p.relative_to(CODE_DIR).as_posix()}")
    z.writestr("Documentation_template.md", doc)
names = zipfile.ZipFile(zpath).namelist()
assert {"output/matching_results.tsv", "output/candidate_pairs.tsv", "Documentation_template.md"} <= set(names)

# leave ONLY the zip in /kaggle/working. Never outside Kaggle: there BASE is the
# current directory, which could be a repo checkout.
if IS_KAGGLE and BASE == Path("/kaggle/working"):
    for p in BASE.iterdir():
        if p.resolve() == zpath.resolve():
            continue
        shutil.rmtree(p) if p.is_dir() else p.unlink()
    print("Output folder now contains:", sorted(x.name for x in BASE.iterdir()))
else:
    print("Not on Kaggle: left the working files in place (cleanup only runs in /kaggle/working).")
print(zpath, f"{zpath.stat().st_size / 1e6:.1f} MB")
for n in names:
    print("  ", n)
if cfg["dev_states"]:
    print("*** DEV zip (partial states only) - NOT for submission. Set DEV = False for the real run. ***")
else:
    print("Leaderboard: upload output/matching_results.tsv from inside this zip. Final package: the zip itself.")
print()
print(doc[:1200])